#TextMorph


## 1. Environment Setup

In [ ]:
!apt-get update -q
!apt-get install -y postgresql postgresql-contrib libpq-dev python3-dev build-essential -q

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://ngrok-agent.s3.amazonaws.com buster InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,264 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,623 kB]
Get:14 http://security.ubuntu.com/ub

## 2. Start PostgreSQL Service

In [ ]:
!service postgresql start


 * Starting PostgreSQL 14 database server
   ...done.


## 3. Mount Google Drive

In [ ]:
# ── Mount Google Drive & persist PostgreSQL data ──────────────
import os, subprocess, shutil
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DB_DIR = '/content/drive/MyDrive/postgres_milestone_db'
PG_DATA_DIR  = '/var/lib/postgresql/14/main'
BACKUP_PATH  = '/content/drive/MyDrive/postgres_milestone_db/milestone1_users.sql'

os.makedirs(DRIVE_DB_DIR, exist_ok=True)

# Start PostgreSQL first
subprocess.run(['service', 'postgresql', 'start'], capture_output=True)

# Set password
subprocess.run(['sudo','-u','postgres','psql','-c',
                "ALTER USER postgres PASSWORD 'postgres';"], capture_output=True)

# Create DB if not exists
result = subprocess.run(['sudo','-u','postgres','psql','-lqt'], capture_output=True, text=True)
if 'milestone1_users' not in result.stdout:
    subprocess.run(['sudo','-u','postgres','psql','-c','CREATE DATABASE milestone1_users;'], capture_output=True)
    print('✅ Database created fresh')

# ── Restore from Drive backup if it exists ────────────────────
if os.path.exists(BACKUP_PATH):
    print('📂 Found Drive backup — restoring...')
    subprocess.run(
        f'sudo -u postgres psql milestone1_users < "{BACKUP_PATH}"',
        shell=True, capture_output=True
    )
    print('✅ Database restored from Google Drive!')
else:
    print('ℹ️ No backup found — starting fresh (will save on next run)')

print('\n✅ PostgreSQL ready. Drive connected.')
print(f'📁 Backups saved to: {DRIVE_DB_DIR}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Database created fresh
ℹ️ No backup found — starting fresh (will save on next run)

✅ PostgreSQL ready. Drive connected.
📁 Backups saved to: /content/drive/MyDrive/postgres_milestone_db


## 4. Configure Database

In [ ]:
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"
!sudo -u postgres psql -c "CREATE DATABASE IF NOT EXISTS milestone1_users;" 2>/dev/null || sudo -u postgres psql -c "SELECT 1 FROM pg_database WHERE datname='milestone1_users'" | grep -q 1 || sudo -u postgres psql -c "CREATE DATABASE milestone1_users;"


ALTER ROLE


### 4.1 Verify Database

In [ ]:
import subprocess
result = subprocess.run(['sudo', '-u', 'postgres', 'psql', '-lqt'], capture_output=True, text=True)
if 'milestone1_users' not in result.stdout:
    subprocess.run(['sudo', '-u', 'postgres', 'psql', '-c', 'CREATE DATABASE milestone1_users;'])
    print('Database created.')
else:
    print('Database already exists.')

Database already exists.


## 5. Test Connection

In [ ]:
import psycopg2
conn = psycopg2.connect(dbname='milestone1_users', user='postgres', password='postgres', host='localhost')
print('✅ PostgreSQL Connected Successfully!')
conn.close()

✅ PostgreSQL Connected Successfully!


## 6. Install Dependencies

In [ ]:
!pip install streamlit psycopg2-binary bcrypt pyjwt watchdog pyngrok -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 126.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 123.9 MB/s eta 0:00:00


### 6.2 Full Packages

In [ ]:
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok nltk streamlit-option-menu plotly textstat PyPDF2 transformers torch sentencepiece accelerate pandas "bitsandbytes>=0.40.0" psycopg2-binary wordcloud matplotlib Pillow -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 35.3 MB/s eta 0:00:00


## 7. App Modules
### 7.1 Readability

In [ ]:
%%writefile text_readability.py
import textstat

class ReadabilityAnalyzer:
    def __init__(self, text):
        self.text = text
        self.num_sentences = textstat.sentence_count(text)
        self.num_words = textstat.lexicon_count(text, removepunct=True)
        self.num_syllables = textstat.syllable_count(text)
        self.complex_words = textstat.difficult_words(text)
        self.char_count = textstat.char_count(text)

    def get_all_metrics(self):
        return {
            "Flesch Reading Ease": textstat.flesch_reading_ease(self.text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(self.text),
            "SMOG Index": textstat.smog_index(self.text),
            "Gunning Fog": textstat.gunning_fog(self.text),
            "Coleman-Liau": textstat.coleman_liau_index(self.text)
        }


Writing text_readability.py


### 7.2 Engine

In [ ]:
%%writefile engine.py
import os
import torch
import streamlit as st
import nltk
from nltk.tokenize import sent_tokenize


try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

TRANSFORMERS_AVAILABLE = False
BNB_AVAILABLE = False
try:
    from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
    TRANSFORMERS_AVAILABLE = True
    try:
        from transformers import BitsAndBytesConfig
        BNB_AVAILABLE = True
    except ImportError:
        pass
except ImportError:
    TRANSFORMERS_AVAILABLE = False

LANG_CODES = {
    "English": "eng_Latn", "Hindi": "hin_Deva", "Tamil": "tam_Taml",
    "Kannada": "kan_Knda", "Telugu": "tel_Telu", "Marathi": "mar_Deva",
    "Bengali": "ben_Beng"
}

@st.cache_resource(show_spinner=False)
def load_translation_model():
    """Load NLLB multilingual translation model."""
    if not TRANSFORMERS_AVAILABLE:
        return None
    try:
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
        model = AutoModelForSeq2SeqLM.from_pretrained(
            "facebook/nllb-200-distilled-600M",
            device_map="auto"
        )
        return {"tokenizer": tokenizer, "model": model}
    except Exception as e:
        print(f"Translation model load failed: {e}")
        return None

def translate_text(text, target_lang, translation_model):
    """Translate text to target language using NLLB model."""
    if target_lang == "English" or not text.strip():
        return text
    lang_code = LANG_CODES.get(target_lang)
    if not lang_code:
        return text
    if translation_model is None:
        return text + f"\n\n[⚠️ Translation model unavailable. Install: pip install transformers]"
    try:
        tokenizer = translation_model["tokenizer"]
        model     = translation_model["model"]
        device    = next(model.parameters()).device
        # Split into chunks to handle long text
        sentences = text.split(". ")
        chunks, curr = [], []
        curr_len = 0
        for s in sentences:
            wlen = len(s.split())
            if curr_len + wlen > 200 and curr:
                chunks.append(". ".join(curr) + ".")
                curr, curr_len = [s], wlen
            else:
                curr.append(s)
                curr_len += wlen
        if curr:
            chunks.append(". ".join(curr))
        translated_parts = []
        for chunk in chunks:
            if not chunk.strip():
                continue
            inputs = tokenizer(chunk, return_tensors="pt",
                               padding=True, truncation=True,
                               max_length=512).to(device)
            target_id = tokenizer.convert_tokens_to_ids(lang_code)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    forced_bos_token_id=target_id,
                    max_new_tokens=512,
                    num_beams=4,
                    early_stopping=True
                )
            translated_parts.append(
                tokenizer.decode(outputs[0], skip_special_tokens=True)
            )
        return " ".join(translated_parts)
    except Exception as e:
        return text + f"\n\n[Translation error: {str(e)}]"


@st.cache_resource(show_spinner=False)
def load_summarization_models(quantization_level="4-bit"):
    models = {}
    if not TRANSFORMERS_AVAILABLE:
        return models
    kwargs = {"device_map": "auto"}
    if BNB_AVAILABLE:
        if quantization_level == "8-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
        elif quantization_level == "4-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    try:
        models['bart'] = {
            'tokenizer': AutoTokenizer.from_pretrained("sshleifer/distilbart-cnn-12-6"),
            'model': AutoModelForSeq2SeqLM.from_pretrained("sshleifer/distilbart-cnn-12-6", **kwargs)
        }
    except Exception as e:
        print(f"BART load failed: {e}"); models['bart'] = None
    try:
        models['pegasus'] = {
            'tokenizer': AutoTokenizer.from_pretrained("google/pegasus-cnn_dailymail"),
            'model': AutoModelForSeq2SeqLM.from_pretrained("google/pegasus-cnn_dailymail", **kwargs)
        }
    except Exception as e:
        print(f"Pegasus load failed: {e}"); models['pegasus'] = None
    try:
        for t5m in ["google/flan-t5-base", "google/flan-t5-small"]:
            try:
                models['flan-t5'] = {
                    'tokenizer': AutoTokenizer.from_pretrained(t5m),
                    'model': AutoModelForSeq2SeqLM.from_pretrained(t5m, **kwargs)
                }
                break
            except Exception:
                continue
        if 'flan-t5' not in models:
            models['flan-t5'] = None
    except Exception as e:
        print(f"FLAN-T5 load failed: {e}"); models['flan-t5'] = None
    return models

@st.cache_resource(show_spinner=False)
def load_paraphrase_models(quantization_level="4-bit"):
    models = {}
    if not TRANSFORMERS_AVAILABLE:
        return models
    kwargs = {"device_map": "auto"}
    if BNB_AVAILABLE:
        if quantization_level == "8-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
        elif quantization_level == "4-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    try:
        try:
            models['flan_t5'] = {
                'tokenizer': AutoTokenizer.from_pretrained("Vamsi/T5_Paraphrase_Paws"),
                'model': AutoModelForSeq2SeqLM.from_pretrained("Vamsi/T5_Paraphrase_Paws", **kwargs)
            }
        except Exception:
            models['flan_t5'] = {
                'tokenizer': AutoTokenizer.from_pretrained("google/flan-t5-small"),
                'model': AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small", **kwargs)
            }
        try:
            models['bart'] = {
                'tokenizer': AutoTokenizer.from_pretrained("eugenesiow/bart-paraphrase"),
                'model': AutoModelForSeq2SeqLM.from_pretrained("eugenesiow/bart-paraphrase", **kwargs)
            }
        except Exception:
            models['bart'] = None
    except Exception:
        pass
    return models

def simple_text_summarization(text, summary_length):
    try:
        sentences = sent_tokenize(text)
        if len(sentences) <= 2:
            return text[:100] + "..." if len(text) > 100 else text
        if summary_length == "Short":
            return " ".join(sentences[:max(2, len(sentences) // 4)])
        elif summary_length == "Medium":
            return " ".join(sentences[:max(3, len(sentences) // 2)])
        else:
            return " ".join(sentences[:max(4, int(len(sentences) * 0.75))])
    except:
        return text[:150] + "..." if len(text) > 150 else text

def _detect_hallucination(original_text, generated_text):
    from collections import Counter
    gen_words = generated_text.split()
    orig_words = set(original_text.lower().split())
    if len(gen_words) < 3:
        return True
    word_counts = Counter(w.lower().strip(".,!?();:'\"") for w in gen_words)
    most_common_count = word_counts.most_common(1)[0][1] if word_counts else 0
    if most_common_count > len(gen_words) * 0.5 and len(gen_words) > 20:
        return True
    gen_clean = [w.lower().strip(".,!?();:'\"") for w in gen_words]
    novel_words = [w for w in gen_clean if w not in orig_words and len(w) > 3]
    if len(novel_words) > len(gen_words) * 0.85 and len(gen_words) > 30:
        return True
    return False

def local_summarize(text, summary_length, model_type, models_dict, target_lang="English"):
    model_key = model_type.lower()
    if model_key not in models_dict or models_dict[model_key] is None:
        st.warning(f"⚠️ {model_type} not available. Using fallback.")
        return simple_text_summarization(text, summary_length)

    model_info       = models_dict[model_key]
    tokenizer        = model_info['tokenizer']
    model            = model_info['model']
    input_word_count = len(text.split())

    # ── FIX: scale summary length proportionally to input word count ──────────
    # Previously used token-count with hard ceilings that capped short texts
    # (e.g. 136 words → only 9-word output). Now we scale directly from words.
    length_config = {
        "Short": {
            "max_new_tokens": max(60,  int(input_word_count * 0.30)),
            "min_new_tokens": max(30,  int(input_word_count * 0.15)),
        },
        "Medium": {
            "max_new_tokens": max(120, int(input_word_count * 0.55)),
            "min_new_tokens": max(60,  int(input_word_count * 0.30)),
        },
        "Long": {
            "max_new_tokens": max(220, int(input_word_count * 0.80)),
            "min_new_tokens": max(100, int(input_word_count * 0.50)),
        },
    }
    config = length_config.get(summary_length, length_config["Medium"])

    # Safety: min must always be less than max
    config["min_new_tokens"] = min(config["min_new_tokens"], config["max_new_tokens"] - 10)
    config["min_new_tokens"] = max(config["min_new_tokens"], 10)
    # ──────────────────────────────────────────────────────────────────────────

    if model_key == 'flan-t5':
        prompts = {
            "Short":  f"Write a brief 2-3 sentence summary: {text}",
            "Medium": f"Write a detailed summary covering main points: {text}",
            "Long":   f"Write a comprehensive summary covering all key points: {text}"
        }
        prompt = prompts.get(summary_length, prompts["Medium"])
    else:
        prompt = text

    try:
        with st.spinner(f"🧠 {model_type} generating summary..."):
            device = next(model.parameters()).device
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                               max_length=1024, padding=True).to(device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=config["max_new_tokens"],
                    min_new_tokens=config["min_new_tokens"],
                    num_beams=4,
                    length_penalty={"Short": 0.8, "Medium": 1.0, "Long": 1.8}.get(summary_length, 1.0),
                    no_repeat_ngram_size=3,
                    early_stopping=True,
                    use_cache=True,
                    repetition_penalty=1.5,
                )
            summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
            if _detect_hallucination(text, summary) or not summary.strip():
                summary = simple_text_summarization(text, summary_length)
            return summary
    except Exception as e:
        st.error(f"❌ {model_type} error: {str(e)}")
        return simple_text_summarization(text, summary_length)

def apply_fallback_paraphrasing(text, complexity):
    subs = {
        "Simple":   {"utilize":"use","facilitate":"help","fundamental":"basic","however":"but","moreover":"also"},
        "Neutral":  {"use":"utilize","help":"assist","basic":"fundamental","but":"however","also":"furthermore"},
        "Advanced": {"use":"leverage","help":"facilitate","basic":"foundational","but":"nevertheless","also":"moreover","show":"demonstrate","important":"paramount"},
    }
    sub_dict = subs.get(complexity, subs["Neutral"])
    result = []
    for word in text.split():
        clean = word.strip(".,!?();:'\"").lower()
        if clean in sub_dict:
            new = sub_dict[clean]
            if word[0].isupper(): new = new.capitalize()
            result.append(new)
        else:
            result.append(word)
    return " ".join(result)

def paraphrase_with_model(text, complexity, style, model_type, models_dict, target_lang="English"):
    model_key = model_type.lower().replace('-', '_')
    try:
        model_info = models_dict.get(model_key)
        if model_info is None:
            return apply_fallback_paraphrasing(text, complexity)
        tokenizer = model_info['tokenizer']
        model     = model_info['model']
        device    = next(model.parameters()).device
        sentences = sent_tokenize(text)
        chunks, curr, curr_len = [], [], 0
        for s in sentences:
            slen = len(s.split())
            if curr_len + slen > 80 and curr:
                chunks.append(" ".join(curr)); curr = [s]; curr_len = slen
            else:
                curr.append(s); curr_len += slen
        if curr: chunks.append(" ".join(curr))
        results = []
        for chunk in chunks:
            token_count = len(tokenizer.encode(chunk))
            if model_key == 'flan_t5':
                prompt = f"paraphrase the following text using different words and sentence structure: {chunk} </s>"
            else:
                prompt = f"paraphrase: {chunk}"
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                               max_length=512, padding="max_length").to(device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max(150, int(token_count * 1.5)),
                    min_new_tokens=max(10,  int(token_count * 0.6)),
                    num_beams=1,
                    no_repeat_ngram_size=3,
                    repetition_penalty=1.8,
                    use_cache=True
                )
            paraphrased = tokenizer.decode(outputs[0], skip_special_tokens=True)
            results.append(paraphrased if len(paraphrased.strip()) > 10 else chunk)
        return " ".join(results) or apply_fallback_paraphrasing(text, complexity)
    except Exception as e:
        st.error(f"❌ Paraphrasing error ({model_type}): {str(e)}")
        return apply_fallback_paraphrasing(text, complexity)


Writing engine.py


### 6.3 Extra Packages

In [ ]:
!pip install plotly PyPDF2 textstat py-readability-metrics

### 7.3 app.py — Milestone 4 (v6 - Chat feature)


In [ ]:
%%writefile app.py

import streamlit as st
import psycopg2
import jwt
import datetime
import bcrypt
import os
import re
import time
import smtplib
import secrets
import hmac
import hashlib
import struct
import random
import base64
import html as _html
from io import BytesIO

# ── Create config.toml for Streamlit theme ────────────────
import os as _os
_os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as _f:
    _f.write("""
[theme]
base = "light"
primaryColor = "#7C3AED"
backgroundColor = "#EDE9FE"
secondaryBackgroundColor = "#F1F5F9"
textColor = "#1F2937"
font = "sans serif"
""")
try:
    from wordcloud import WordCloud
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    WORDCLOUD_AVAILABLE = True
except ImportError:
    WORDCLOUD_AVAILABLE = False
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from streamlit_option_menu import option_menu
import plotly.graph_objects as go
import PyPDF2
import textstat
import pandas as pd
from text_readability import ReadabilityAnalyzer
import engine

EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
SECRET_KEY     = os.getenv("JWT_SECRET", "dev_secret_key")
ADMIN_EMAIL_ID = os.getenv("ADMIN_EMAIL_ID")
ADMIN_PASSWORD = os.getenv("ADMIN_PASSWORD")
EMAIL_ADDRESS  = os.getenv("EMAIL_ID")
ALGORITHM      = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30
OTP_EXPIRY_MINUTES = 10
MAX_LOGIN_ATTEMPTS = 3
LOCKOUT_SECONDS    = 300

DB_NAME     = os.getenv("DB_NAME",     "milestone1_users")
DB_USER     = os.getenv("DB_USER",     "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "postgres")
DB_HOST     = os.getenv("DB_HOST",     "localhost")
DB_PORT     = os.getenv("DB_PORT",     "5432")

SUPPORTED_LANGUAGES = ["English", "Hindi", "Tamil", "Kannada", "Telugu", "Marathi", "Bengali"]

st.set_page_config(page_title="TEXTMORPH", page_icon="💡", layout="wide",
                   initial_sidebar_state="expanded")

# ── FIX: Load models only once, guard with session_state flag ──
if 'models_loaded' not in st.session_state:
    st.session_state['models_loaded'] = False

if not st.session_state['models_loaded']:
    with st.spinner("Loading AI models..."):
        st.session_state.summarization_models = engine.load_summarization_models()
        st.session_state.paraphrase_models    = engine.load_paraphrase_models()
        st.session_state.translation_model    = engine.load_translation_model()
        st.session_state['models_loaded'] = True

SECURITY_QUESTIONS = [
    "What is your pet's name?",
    "What is your mother's maiden name?",
    "Who was your favorite teacher?",
    "What city were you born in?",
    "What was the name of your first school?"
]
for key, val in [("jwt_token", None), ("page", "login"), ("username", None), ("theme", "light")]:
    if key not in st.session_state: st.session_state[key] = val

# ══════════════════════════════════════════════════════════
# BASE CSS  (light theme)
# ══════════════════════════════════════════════════════════
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600&display=swap');

#MainMenu {visibility:hidden;}
footer    {visibility:hidden;}

.stApp {
  background: linear-gradient(135deg, #EDE9FE 0%, #DDD6FE 100%) !important;
  font-family: 'Inter', sans-serif;
  color: #1F2937;
}

[data-testid="stHeader"] {
  background: linear-gradient(135deg, #EDE9FE 0%, #DDD6FE 100%) !important;
  border-bottom: 1px solid #C4B5FD !important;
  box-shadow: none !important;
}

[data-testid="stSidebar"] {
  background: #FFFFFF !important;
  border-right: 2px solid #C4B5FD !important;
}
[data-testid="stSidebar"] * { color: #1F2937 !important; }

[data-testid="stVerticalBlockBorderWrapper"] {
  background: #FFFFFF !important;
  border: 1px solid #C4B5FD !important;
  border-radius: 12px !important;
  box-shadow: 0 2px 12px rgba(139,92,246,0.10);
  padding: 10px;
}

.stButton>button[kind="primary"] {
  background: linear-gradient(135deg, #7C3AED, #6D28D9) !important;
  color: #FFFFFF !important;
  border-radius: 8px;
  border: none;
  font-weight: 600;
  box-shadow: 0 2px 10px rgba(109,40,217,0.40);
}
.stButton>button[kind="secondary"] {
  background: #F1F5F9 !important;
  color: #6D28D9 !important;
  border-radius: 8px;
  border: 1px solid #C4B5FD !important;
}

.stTextInput input,
.stTextArea textarea,
.stTextInput>div>div>input,
.stTextArea>div>div>textarea {
  background: #FFFFFF !important;
  border: 1.5px solid #C4B5FD !important;
  border-radius: 8px !important;
  color: #1F2937 !important;
  -webkit-text-fill-color: #1F2937 !important;
  caret-color: #6D28D9 !important;
}
.stTextInput input::placeholder,
.stTextArea textarea::placeholder {
  color: #9CA3AF !important;
  -webkit-text-fill-color: #9CA3AF !important;
}

.stTextInput label, .stTextArea label, .stSelectbox label,
label[data-testid="stWidgetLabel"] > div > p {
  color: #374151 !important;
  font-weight: 500 !important;
}

.stTabs [data-baseweb="tab-list"] { background: transparent !important; }
.stTabs [data-baseweb="tab"] { color: #6B7280 !important; background: transparent !important; }
.stTabs [aria-selected="true"] {
  color: #6D28D9 !important;
  border-bottom: 2px solid #8B5CF6;
  font-weight: 600;
  background: #EDE9FE !important;
}

[data-testid="metric-container"] {
  background: #FFFFFF;
  border: 1px solid #C4B5FD;
  border-radius: 10px;
  padding: 12px;
}
[data-testid="stMetricValue"] { color: #6D28D9 !important; }
[data-testid="stMetricLabel"] { color: #6B7280 !important; }

div.page-header {
  display: flex !important;
  align-items: center !important;
  gap: 14px !important;
  background: linear-gradient(135deg, #7C3AED, #A78BFA) !important;
  border-radius: 14px !important;
  padding: 18px 22px !important;
  margin-bottom: 24px !important;
  box-shadow: 0 4px 20px rgba(109,40,217,0.30) !important;
}

.stat-card {
  background: #FFFFFF;
  border: 1px solid #C4B5FD;
  border-top: 3px solid #8B5CF6;
  border-radius: 12px;
  padding: 18px;
  text-align: center;
}
.stat-num   { font-size: 26px; font-weight: 700; color: #6D28D9; }
.stat-label { font-size: 12px; color: #6B7280; margin-top: 4px; }

.hist-card {
  background: #FFFFFF;
  border: 1px solid #C4B5FD;
  border-radius: 10px;
  padding: 14px 18px;
  margin-bottom: 10px;
}
.result-box-original {
  background: #F1F5F9;
  color: #1F2937 !important;
  border-left: 4px solid #A78BFA;
  border-radius: 8px;
  padding: 14px 16px;
  min-height: 120px;
  font-size: 14px;
  line-height: 1.6;
}
.result-box-output {
  background: #F1F5F9;
  color: #1F2937 !important;
  border-left: 4px solid #8E4585;
  border-radius: 8px;
  padding: 14px 16px;
  min-height: 120px;
  font-size: 14px;
  line-height: 1.6;
}

/* ── SETTINGS PANEL: pure CSS box, no phantom widget ── */
.settings-panel-wrap {
  background: #F1F5F9;
  border: 1px solid #C4B5FD;
  border-radius: 10px;
  padding: 16px 18px;
  margin-bottom: 12px;
}

.sidebar-logo  { text-align: center; padding: 20px 0 12px; }
.app-name      { font-size: 18px; font-weight: 700; color: #6D28D9 !important; margin-top: 8px; }
.user-name     { font-size: 13px; color: #1F2937 !important; margin-top: 4px; }
.user-email    { font-size: 11px; color: #6B7280 !important; margin-top: 2px; }

.admin-row {
  background: #F1F5F9;
  border: 1px solid #C4B5FD;
  border-radius: 8px;
  padding: 10px 14px;
  margin-bottom: 6px;
}

.streamlit-expanderHeader {
  background: #EDE9FE !important;
  color: #6D28D9 !important;
  border-radius: 8px;
}

/* ── DATAFRAME: light theme ── */
.stDataFrame thead th,
[data-testid="stDataFrame"] thead th {
  background: linear-gradient(135deg, #7C3AED, #A78BFA) !important;
  color: #FFFFFF !important;
  font-weight: 600 !important;
}

.stProgress > div > div { background: #8E4585 !important; }

::-webkit-scrollbar       { width: 6px; height: 6px; }
::-webkit-scrollbar-track { background: #EDE9FE; }
::-webkit-scrollbar-thumb { background: #C4B5FD; border-radius: 3px; }

.stSelectbox > div > div {
  background: #FFFFFF !important;
  border: 1.5px solid #C4B5FD !important;
  color: #1F2937 !important;
  border-radius: 8px !important;
}

hr { border-color: #C4B5FD !important; }
.stAlert { border-radius: 8px !important; }
.llm-bg-orbs, .llm-orb { display: none !important; }

button[data-testid="stBaseButton-primary"],
.stButton > button,
div[data-testid="stButton"] > button {
  background: linear-gradient(135deg, #8E4585, #6D28D9) !important;
  background-color: #6D28D9 !important;
  color: #FFFFFF !important;
  border: none !important;
}
.stTabs [data-baseweb="tab-highlight"] { background-color: #8B5CF6 !important; }

p, div, span, li { color: #1F2937; }
h1, h2, h3, h4, h5, h6 { color: #4C1D95 !important; }

.stTextArea, .stFileUploader, .stSelectbox, .stButton, .stMarkdown { margin-bottom: 8px !important; }
[data-testid="stVerticalBlock"] > div { margin-bottom: 6px !important; }
</style>
""", unsafe_allow_html=True)

# ── Dark theme overrides ───────────────────────────────────
_cur_theme = st.session_state.get("theme", "dark")
if _cur_theme == "dark":
    st.markdown("""<style>
/* ===== DARK THEME OVERRIDES ===== */
.stApp { background: #1A1A1A !important; color: #ECECEC !important; }

[data-testid="stHeader"] {
  background: #1A1A1A !important;
  border-bottom: 1px solid #333333 !important;
}
[data-testid="stSidebar"] { background: #212121 !important; border-right: 2px solid #333333 !important; }
[data-testid="stSidebar"] * { color: #E2D9F3 !important; }
.app-name   { color: #C4B5FD !important; }
.user-name  { color: #E2D9F3 !important; }
.user-email { color: #9C8BBF !important; }

[data-testid="stVerticalBlockBorderWrapper"] {
  background: #2A2A2A !important;
  border: 1px solid #3D3D3D !important;
  box-shadow: 0 2px 12px rgba(0,0,0,0.40) !important;
}

.stTextInput input, .stTextArea textarea,
.stTextInput>div>div>input, .stTextArea>div>div>textarea {
  background: #2A2A2A !important;
  border: 1.5px solid #444444 !important;
  color: #E2D9F3 !important;
  -webkit-text-fill-color: #E2D9F3 !important;
  caret-color: #A78BFA !important;
}
.stTextInput input::placeholder, .stTextArea textarea::placeholder {
  color: #6B5F8A !important;
  -webkit-text-fill-color: #6B5F8A !important;
}

.stTextInput label, .stTextArea label, .stSelectbox label,
label[data-testid="stWidgetLabel"] > div > p { color: #C4B5FD !important; }

.stTabs [data-baseweb="tab-list"] { background: transparent !important; }
.stTabs [data-baseweb="tab"] { color: #9C8BBF !important; background: transparent !important; }
.stTabs [aria-selected="true"] {
  color: #C4B5FD !important;
  background: #2D1F55 !important;
  border-bottom: 2px solid #8B5CF6 !important;
}
.stApp .stTabs [aria-selected="true"],
.stApp div[data-baseweb="tab-list"] button[aria-selected="true"] {
  background: #2D1F55 !important;
  color: #C4B5FD !important;
  border-bottom: 2px solid #8B5CF6 !important;
}

[data-testid="metric-container"] { background: #2A2A2A !important; border: 1px solid #3D3D3D !important; }
[data-testid="stMetricValue"] { color: #C4B5FD !important; }
[data-testid="stMetricLabel"] { color: #9C8BBF !important; }
[data-testid="stMetricDelta"] { color: #A78BFA !important; }

.stat-card { background: #2A2A2A !important; border: 1px solid #3D3D3D !important; border-top: 3px solid #6D28D9 !important; }
.stat-num   { color: #C4B5FD !important; }
.stat-label { color: #9C8BBF !important; }

.hist-card { background: #2A2A2A !important; border: 1px solid #3D3D3D !important; }
.hist-card:hover { border-color: #8B5CF6 !important; }

.result-box-original { background: #2A2A2A !important; color: #E2D9F3 !important; border-left: 4px solid #A78BFA !important; }
.result-box-output   { background: #2A2A2A !important; color: #E2D9F3 !important; border-left: 4px solid #8E4585 !important; }

/* ── SETTINGS PANEL dark ── */
.settings-panel-wrap {
  background: #2A2A2A !important;
  border: 1px solid #3D3D3D !important;
}

.admin-row { background: #2A2A2A !important; border: 1px solid #3D3D3D !important; }

.stSelectbox > div > div {
  background: #2A2A2A !important;
  border: 1.5px solid #444444 !important;
  color: #E2D9F3 !important;
}
.stSelectbox [data-baseweb="select"] div { color: #E2D9F3 !important; }

.stButton>button[kind="secondary"] { background: #2A2A2A !important; color: #C4B5FD !important; border-color: #444444 !important; }

p, div, span, label { color: #E2D9F3; }
h1, h2, h3, h4, h5, h6 { color: #C4B5FD !important; }

/* ─── DARK TABLE — COMPLETE FIX ─── */
/* Force the entire dataframe area dark */
[data-testid="stDataFrame"],
[data-testid="stDataFrame"] > div,
[data-testid="stDataFrame"] iframe,
.stDataFrame {
  background: #1E1E1E !important;
  background-color: #1E1E1E !important;
  color: #ECECEC !important;
  color-scheme: dark !important;
}

/* All text inside dataframe must be light */
[data-testid="stDataFrame"] *,
[data-testid="stDataFrame"] div,
[data-testid="stDataFrame"] span,
[data-testid="stDataFrame"] p {
  color: #ECECEC !important;
  background-color: transparent !important;
}

/* Header row — keep purple gradient */
[data-testid="stDataFrame"] thead th,
.stDataFrame thead th {
  background: linear-gradient(135deg, #4C1D95, #7C3AED) !important;
  background-color: #4C1D95 !important;
  color: #FFFFFF !important;
  font-weight: 600 !important;
  border-color: #3D3D3D !important;
}

/* Body cells */
[data-testid="stDataFrame"] tbody td,
.stDataFrame tbody tr td {
  background-color: #1E1E1E !important;
  color: #ECECEC !important;
  border-color: #3D3D3D !important;
}
[data-testid="stDataFrame"] tbody tr:nth-child(even) td {
  background-color: #272727 !important;
  color: #ECECEC !important;
}
[data-testid="stDataFrame"] tbody tr:hover td {
  background-color: #333333 !important;
  color: #FFFFFF !important;
}

/* Arrow/glide grid canvas fix for dark */
[data-testid="stDataFrame"] canvas {
  filter: invert(0.88) hue-rotate(180deg);
}

/* Scrollbar inside dataframe dark */
[data-testid="stDataFrame"] ::-webkit-scrollbar-track { background: #1E1E1E !important; }
[data-testid="stDataFrame"] ::-webkit-scrollbar-thumb { background: #444444 !important; }

.streamlit-expanderHeader { background: #2A2A2A !important; color: #C4B5FD !important; }

::-webkit-scrollbar-track { background: #1A1A1A !important; }
::-webkit-scrollbar-thumb { background: #444444 !important; }

hr { border-color: #3D3D3D !important; }
.stAlert { background: #2A2A2A !important; border-color: #3D3D3D !important; }
[data-testid="stAlert"] { background: #2A2A2A !important; border-color: #444444 !important; color: #ECECEC !important; }
[data-testid="stAlert"] * { color: #ECECEC !important; }

.page-header {
  background: linear-gradient(135deg, #4C1D95, #7C3AED) !important;
  box-shadow: 0 4px 20px rgba(76,29,149,0.50) !important;
}

button[data-testid="stBaseButton-primary"],
.stButton > button,
div[data-testid="stButton"] > button {
  background: linear-gradient(135deg, #8E4585, #6D28D9) !important;
  background-color: #6D28D9 !important;
  color: #FFFFFF !important;
  border: none !important;
}
.stTabs [data-baseweb="tab-highlight"] { background-color: #8B5CF6 !important; }

[data-testid="stSlider"] [role="slider"] { background-color: #8B5CF6 !important; border-color: #6D28D9 !important; }

/* File uploader dark */
.stApp [data-testid="stFileUploader"],
.stApp [data-testid="stFileUploader"] > div,
.stApp [data-testid="stFileUploadDropzone"],
[data-testid="stFileUploadDropzone"],
[data-testid="stFileUploadDropzone"] > div,
[data-testid="stFileUploadDropzone"] > div > div {
  background: #2A2A2A !important;
  background-color: #2A2A2A !important;
  border: 1.5px dashed #555555 !important;
  color: #ECECEC !important;
}
[data-testid="stFileUploadDropzone"] * { background: #2A2A2A !important; color: #ECECEC !important; }

/* Eye button fix */
div[data-testid="stTextInput"] button,
div[data-baseweb="input"] button,
div[data-baseweb="base-input"] button {
  background: transparent !important;
  background-color: transparent !important;
  background-image: none !important;
  border: none !important;
  box-shadow: none !important;
  color: #6B7280 !important;
}
</style>
""", unsafe_allow_html=True)

# ── DB connection ──────────────────────────────────────────
def get_connection():
    return psycopg2.connect(dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD,
                            host=DB_HOST, port=DB_PORT)

def create_tables():
    conn = get_connection(); cur = conn.cursor()
    cur.execute("""CREATE TABLE IF NOT EXISTS users (
        id SERIAL PRIMARY KEY, username VARCHAR(100) UNIQUE NOT NULL,
        email VARCHAR(150) UNIQUE NOT NULL, password TEXT NOT NULL,
        security_question TEXT NOT NULL, security_answer TEXT NOT NULL,
        created_at TIMESTAMP DEFAULT NOW());""")
    cur.execute("""CREATE TABLE IF NOT EXISTS password_history (
        id SERIAL PRIMARY KEY, email VARCHAR(150) NOT NULL, password TEXT NOT NULL,
        set_at TIMESTAMP DEFAULT NOW(),
        FOREIGN KEY (email) REFERENCES users(email) ON DELETE CASCADE);""")
    cur.execute("""CREATE TABLE IF NOT EXISTS login_attempts (
        email VARCHAR(150) PRIMARY KEY, attempts INTEGER DEFAULT 0,
        last_attempt DOUBLE PRECISION DEFAULT 0);""")
    cur.execute("""CREATE TABLE IF NOT EXISTS user_history (
        id SERIAL PRIMARY KEY, username VARCHAR(100) NOT NULL,
        action_type VARCHAR(50) NOT NULL, model_used VARCHAR(100),
        settings TEXT, input_snippet TEXT, output_snippet TEXT,
        language VARCHAR(50) DEFAULT 'English',
        word_count_in INTEGER DEFAULT 0, word_count_out INTEGER DEFAULT 0,
        char_count_in INTEGER DEFAULT 0, char_count_out INTEGER DEFAULT 0,
        created_at TIMESTAMP DEFAULT NOW());""")
    try:
        cur.execute("ALTER TABLE user_history ADD COLUMN IF NOT EXISTS char_count_in INTEGER DEFAULT 0;")
        cur.execute("ALTER TABLE user_history ADD COLUMN IF NOT EXISTS char_count_out INTEGER DEFAULT 0;")
        conn.commit()
    except Exception:
        conn.rollback()
    for stmt in [
        "ALTER TABLE users ADD COLUMN IF NOT EXISTS is_admin BOOLEAN DEFAULT FALSE;",
        "ALTER TABLE users ADD COLUMN IF NOT EXISTS is_locked BOOLEAN DEFAULT FALSE;",
        "ALTER TABLE users ADD COLUMN IF NOT EXISTS avatar TEXT DEFAULT NULL;",
    ]:
        try: cur.execute(stmt); conn.commit()
        except Exception: conn.rollback()
    cur.execute("""CREATE TABLE IF NOT EXISTS user_feedback (
        id SERIAL PRIMARY KEY, username VARCHAR(100) NOT NULL,
        rating INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
        feature VARCHAR(100) DEFAULT 'General',
        comment TEXT,
        created_at TIMESTAMP DEFAULT NOW());""")
    try:
        cur.execute("""
            ALTER TABLE password_history
                DROP CONSTRAINT IF EXISTS password_history_email_fkey;
            ALTER TABLE password_history
                ADD CONSTRAINT password_history_email_fkey
                FOREIGN KEY (email) REFERENCES users(email)
                ON DELETE CASCADE ON UPDATE CASCADE;
        """)
        conn.commit()
    except Exception:
        conn.rollback()
    conn.commit(); cur.close(); conn.close()

create_tables()

# ── History helpers ───────────────────────────────────────
def save_history_to_db(username, action_type, model_used, settings,
                       input_text, output_text, language="English"):
    try:
        conn = get_connection(); cur = conn.cursor()
        inp  = (input_text[:300]  + "...") if len(input_text)  > 300 else input_text
        out  = (output_text[:300] + "...") if len(output_text) > 300 else output_text
        cur.execute("""INSERT INTO user_history
            (username,action_type,model_used,settings,input_snippet,output_snippet,
             language,word_count_in,word_count_out,char_count_in,char_count_out,created_at)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)""",
            (username, action_type, model_used, settings, inp, out, language,
             len(input_text.split()), len(output_text.split()),
             len(input_text), len(output_text),
             datetime.datetime.utcnow()))
        conn.commit(); cur.close(); conn.close()
    except Exception: pass

def get_history_from_db(username, limit=100):
    try:
        conn = get_connection(); cur = conn.cursor()
        cur.execute("""SELECT id,action_type,model_used,settings,input_snippet,
            output_snippet,language,word_count_in,word_count_out,
            COALESCE(char_count_in,0),COALESCE(char_count_out,0),created_at
            FROM user_history WHERE username=%s ORDER BY created_at DESC LIMIT %s""",
            (username, limit))
        data = cur.fetchall(); cur.close(); conn.close(); return data
    except: return []

def get_all_history_from_db(limit=200):
    try:
        conn = get_connection(); cur = conn.cursor()
        cur.execute("""SELECT username,action_type,model_used,settings,input_snippet,
            output_snippet,language,word_count_in,word_count_out,
            COALESCE(char_count_in,0),COALESCE(char_count_out,0),created_at
            FROM user_history ORDER BY created_at DESC LIMIT %s""", (limit,))
        data = cur.fetchall(); cur.close(); conn.close(); return data
    except: return []

def clear_user_history(username):
    try:
        conn = get_connection(); cur = conn.cursor()
        cur.execute("DELETE FROM user_history WHERE username=%s", (username,))
        conn.commit(); cur.close(); conn.close()
    except: pass

# ── Rate limiting ─────────────────────────────────────────
def get_login_attempts(email):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email=%s", (email,))
    data = cur.fetchone(); cur.close(); conn.close()
    return data if data else (0, 0)

def increment_login_attempts(email):
    conn = get_connection(); cur = conn.cursor()
    attempts, _ = get_login_attempts(email); now = time.time()
    cur.execute("""INSERT INTO login_attempts (email,attempts,last_attempt) VALUES (%s,%s,%s)
        ON CONFLICT (email) DO UPDATE SET attempts=%s, last_attempt=%s""",
        (email, attempts+1, now, attempts+1, now))
    conn.commit(); cur.close(); conn.close()

def reset_login_attempts(email):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("DELETE FROM login_attempts WHERE email=%s", (email,))
    conn.commit(); cur.close(); conn.close()

def is_rate_limited(email):
    attempts, last_attempt = get_login_attempts(email)
    if attempts >= MAX_LOGIN_ATTEMPTS:
        elapsed = time.time() - last_attempt
        if elapsed < LOCKOUT_SECONDS: return True, int(LOCKOUT_SECONDS - elapsed)
        else: reset_login_attempts(email)
    return False, 0

# ── User management ───────────────────────────────────────
def check_user_exists_by_email(email):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT id FROM users WHERE email=%s", (email,))
    data = cur.fetchone(); cur.close(); conn.close(); return data is not None

def check_user_exists_by_username(username):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT id FROM users WHERE username=%s", (username,))
    data = cur.fetchone(); cur.close(); conn.close(); return data is not None

def register_user(username, email, password, security_question, security_answer):
    conn = get_connection(); cur = conn.cursor()
    try:
        hp = bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()
        cur.execute("INSERT INTO users (username,email,password,security_question,security_answer) VALUES (%s,%s,%s,%s,%s)",
            (username, email, hp, security_question, security_answer.strip().lower()))
        cur.execute("INSERT INTO password_history (email,password) VALUES (%s,%s)", (email, hp))
        conn.commit(); return True, "Success"
    except Exception as e: conn.rollback(); return False, str(e)
    finally: cur.close(); conn.close()

def authenticate_user(email, password):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT username, password, COALESCE(is_locked,FALSE) FROM users WHERE email=%s", (email,))
    result = cur.fetchone(); cur.close(); conn.close()
    if result:
        uname, hp, locked = result
        if locked: return False, "__LOCKED__"
        if bcrypt.checkpw(password.encode(), hp.encode()):
            reset_login_attempts(email); return True, uname
    increment_login_attempts(email); return False, None

def check_password_reused(email, new_password):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT password FROM password_history WHERE email=%s ORDER BY set_at DESC", (email,))
    for (hp,) in cur.fetchall():
        if bcrypt.checkpw(new_password.encode(), hp.encode()): cur.close(); conn.close(); return True
    cur.close(); conn.close(); return False

def check_is_old_password(email, password):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT password,set_at FROM password_history WHERE email=%s ORDER BY set_at DESC", (email,))
    for hp, set_at in cur.fetchall():
        if bcrypt.checkpw(password.encode(), hp.encode()): cur.close(); conn.close(); return set_at
    cur.close(); conn.close(); return None

def update_password(email, new_password):
    conn = get_connection(); cur = conn.cursor()
    hp = bcrypt.hashpw(new_password.encode(), bcrypt.gensalt()).decode()
    cur.execute("UPDATE users SET password=%s WHERE email=%s", (hp, email))
    cur.execute("INSERT INTO password_history (email,password) VALUES (%s,%s)", (email, hp))
    conn.commit(); cur.close(); conn.close()

def get_security_question(email):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT security_question,security_answer FROM users WHERE email=%s", (email,))
    result = cur.fetchone(); cur.close(); conn.close(); return result

def get_all_users():
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT username,email,created_at,COALESCE(is_admin,FALSE),COALESCE(is_locked,FALSE) FROM users ORDER BY created_at DESC")
    data = cur.fetchall(); cur.close(); conn.close(); return data

def delete_user(email):
    conn = get_connection(); cur = conn.cursor()
    try:
        cur.execute("SELECT username FROM users WHERE email=%s", (email,))
        row = cur.fetchone()
        username = row[0] if row else None
        cur.execute("DELETE FROM password_history WHERE email=%s", (email,))
        cur.execute("DELETE FROM login_attempts WHERE email=%s", (email,))
        if username:
            cur.execute("DELETE FROM user_history WHERE username=%s", (username,))
            cur.execute("DELETE FROM user_feedback WHERE username=%s", (username,))
        cur.execute("DELETE FROM users WHERE email=%s", (email,))
        conn.commit()
    except Exception as e:
        conn.rollback(); raise e
    finally:
        cur.close(); conn.close()

def set_admin_status(email, is_admin):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("UPDATE users SET is_admin=%s WHERE email=%s", (is_admin, email))
    conn.commit(); cur.close(); conn.close()

def set_locked_status(email, is_locked):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("UPDATE users SET is_locked=%s WHERE email=%s", (is_locked, email))
    conn.commit(); cur.close(); conn.close()

def is_account_locked(email):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT COALESCE(is_locked,FALSE) FROM users WHERE email=%s", (email,))
    row = cur.fetchone(); cur.close(); conn.close()
    return row[0] if row else False

def update_user_email(old_email, new_email):
    conn = get_connection(); cur = conn.cursor()
    try:
        cur.execute("UPDATE login_attempts SET email=%s WHERE email=%s", (new_email, old_email))
        cur.execute("UPDATE users SET email=%s WHERE email=%s", (new_email, old_email))
        cur.execute("UPDATE password_history SET email=%s WHERE email=%s", (new_email, old_email))
        conn.commit(); return True, "Email updated"
    except Exception as e: conn.rollback(); return False, str(e)
    finally: cur.close(); conn.close()

def update_user_avatar(email, avatar_b64):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("UPDATE users SET avatar=%s WHERE email=%s", (avatar_b64, email))
    conn.commit(); cur.close(); conn.close()

def get_user_avatar(email):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT avatar FROM users WHERE email=%s", (email,))
    row = cur.fetchone(); cur.close(); conn.close()
    return row[0] if row else None

def get_is_admin_from_db(email):
    try:
        conn = get_connection(); cur = conn.cursor()
        cur.execute("SELECT COALESCE(is_admin, FALSE) FROM users WHERE email=%s", (email,))
        row = cur.fetchone(); cur.close(); conn.close()
        return bool(row[0]) if row else False
    except: return False

# ── Chat helpers ──────────────────────────────────────────
def create_chat_table():
    conn = get_connection(); cur = conn.cursor()
    cur.execute("""CREATE TABLE IF NOT EXISTS chat_messages (
        id SERIAL PRIMARY KEY,
        username VARCHAR(100) NOT NULL,
        role VARCHAR(10) NOT NULL,
        message TEXT NOT NULL,
        model VARCHAR(100) DEFAULT 'FLAN-T5',
        created_at TIMESTAMP DEFAULT NOW()
    );""")
    conn.commit(); cur.close(); conn.close()

create_chat_table()

def save_chat_message(username, role, message, model="FLAN-T5"):
    try:
        conn = get_connection(); cur = conn.cursor()
        cur.execute(
            "INSERT INTO chat_messages (username,role,message,model) VALUES (%s,%s,%s,%s)",
            (username, role, message, model))
        conn.commit(); cur.close(); conn.close()
    except Exception: pass

def get_chat_history(username, limit=100):
    try:
        conn = get_connection(); cur = conn.cursor()
        cur.execute(
            "SELECT role,message,model,created_at FROM chat_messages "
            "WHERE username=%s ORDER BY created_at ASC LIMIT %s",
            (username, limit))
        data = cur.fetchall(); cur.close(); conn.close(); return data
    except: return []

def clear_chat_history(username):
    try:
        conn = get_connection(); cur = conn.cursor()
        cur.execute("DELETE FROM chat_messages WHERE username=%s", (username,))
        conn.commit(); cur.close(); conn.close()
    except: pass

def get_all_chat_users():
    try:
        conn = get_connection(); cur = conn.cursor()
        cur.execute("SELECT DISTINCT username FROM chat_messages ORDER BY username")
        data = [r[0] for r in cur.fetchall()]; cur.close(); conn.close(); return data
    except: return []

def get_chat_history_admin(username, limit=200):
    try:
        conn = get_connection(); cur = conn.cursor()
        cur.execute(
            "SELECT role,message,model,created_at FROM chat_messages "
            "WHERE username=%s ORDER BY created_at ASC LIMIT %s",
            (username, limit))
        data = cur.fetchall(); cur.close(); conn.close(); return data
    except: return []

def save_feedback(username, rating, feature, comment):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("INSERT INTO user_feedback (username,rating,feature,comment) VALUES (%s,%s,%s,%s)",
                (username, rating, feature, comment))
    conn.commit(); cur.close(); conn.close()

def get_all_feedback():
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT username,rating,feature,comment,created_at FROM user_feedback ORDER BY created_at DESC")
    data = cur.fetchall(); cur.close(); conn.close(); return data

# ── JWT ───────────────────────────────────────────────────
def create_access_token(data, expires_minutes=ACCESS_TOKEN_EXPIRE_MINUTES):
    d = data.copy()
    d["exp"] = datetime.datetime.utcnow() + datetime.timedelta(minutes=expires_minutes)
    return jwt.encode(d, SECRET_KEY, algorithm=ALGORITHM)

def verify_token(token):
    try: return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
    except: return None

# ── Validation ────────────────────────────────────────────
def is_valid_email(email):
    return re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', email) is not None

def check_password_strength(password):
    has_upper   = bool(re.search(r"[A-Z]", password))
    has_lower   = bool(re.search(r"[a-z]", password))
    has_digit   = bool(re.search(r"\d",    password))
    has_special = bool(re.search(r'[!@#$%^&*(),.?":{}|<>]', password))
    has_space   = bool(re.search(r"\s",    password))
    if has_space: return "Weak", ["No spaces allowed"]
    is_alphanum = (has_upper or has_lower) and has_digit
    if len(password) >= 8 and is_alphanum and has_special: return "Strong", []
    if len(password) >= 8 and is_alphanum: return "Medium", ["Add special characters"]
    return "Weak", ["Min 8 chars with letters and numbers"]

def get_relative_time(dt):
    if not dt: return "a while ago"
    try:
        if isinstance(dt, str): dt = datetime.datetime.strptime(dt, "%Y-%m-%d %H:%M:%S")
        if hasattr(dt, 'tzinfo') and dt.tzinfo is not None:
            dt = dt.replace(tzinfo=None)
        diff = datetime.datetime.utcnow() - dt
        total_seconds = int(diff.total_seconds())
        if total_seconds < 60:    return "just now"
        elif total_seconds < 3600: return f"{total_seconds // 60}m ago"
        elif total_seconds < 86400: return f"{total_seconds // 3600}h ago"
        elif diff.days > 365: return f"{diff.days//365}y ago"
        elif diff.days > 30:  return f"{diff.days//30}mo ago"
        else: return f"{diff.days}d ago"
    except: return str(dt)

# ── OTP ───────────────────────────────────────────────────
def generate_otp():
    secret    = secrets.token_bytes(20)
    msg       = struct.pack(">Q", int(time.time()))
    h         = hmac.new(secret, msg, hashlib.sha1).digest()
    offset    = h[19] & 0xf
    code      = ((h[offset] & 0x7f) << 24 | (h[offset+1] & 0xff) << 16 |
                 (h[offset+2] & 0xff) << 8  | (h[offset+3] & 0xff))
    return f"{code % 1000000:06d}"

def create_otp_token(otp, email):
    otp_hash = bcrypt.hashpw(otp.encode(), bcrypt.gensalt()).decode()
    payload  = {'otp_hash': otp_hash, 'sub': email, 'type': 'password_reset',
                'iat': datetime.datetime.utcnow(),
                'exp': datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)}
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        if payload.get('sub') != email: return False, "Token mismatch"
        if bcrypt.checkpw(input_otp.encode(), payload['otp_hash'].encode()): return True, "Valid"
        return False, "Invalid OTP"
    except Exception as e: return False, str(e)

def send_otp_email(to_email, otp):
    print(repr(otp))
    email = EMAIL_ADDRESS.strip()
    to_email = to_email.strip()
    if not email or not EMAIL_PASSWORD:
        return False, "Email credentials not set."
    msg = MIMEMultipart()
    msg['From'] = f"LLM S <{email}>"
    msg['To'] = to_email
    msg['Subject'] = "Your Password Reset OTP"
    body = f"""<html><body style="background:#EDE9FE;font-family:Inter,sans-serif;padding:40px;text-align:center;">
    <div style="background:#FFFFFF;border-radius:16px;padding:40px;max-width:480px;margin:auto;border:1px solid #C4B5FD;">
        <div style="font-size:48px;margin-bottom:16px;">🔐</div>
        <h2 style="color:#6D28D9;margin:0 0 12px;">Password Reset OTP</h2>
        <p style="color:#6B7280;margin-bottom:24px;">Use this code for <strong style="color:#8B5CF6;">{to_email}</strong></p>
        <div style="background:#F1F5F9;color:#6D28D9;font-size:36px;font-weight:700;letter-spacing:10px;
                    padding:24px;border-radius:12px;margin:0 0 24px;border:1px solid #C4B5FD;">{otp}</div>
        <p style="color:#6B7280;font-size:13px;">Valid for {OTP_EXPIRY_MINUTES} minutes. Do not share this code.</p>
    </div></body></html>"""
    msg.attach(MIMEText(body, 'html'))
    try:
        s = smtplib.SMTP('smtp.gmail.com', 587)
        s.starttls()
        s.login(email, EMAIL_PASSWORD)
        s.sendmail(email, to_email, msg.as_string())
        s.quit()
        return True, "Sent"
    except Exception as e:
        return False, str(e)

# ── Helpers ───────────────────────────────────────────────
def page_header(icon, title, subtitle=""):
    sub_html = f'<p style="font-size:13px;color:rgba(255,255,255,0.88);margin:3px 0 0;padding:0;">{subtitle}</p>' if subtitle else ""
    st.markdown(f"""
    <div style="display:flex;align-items:center;gap:14px;
        background:linear-gradient(135deg,#7C3AED,#A78BFA);
        border-radius:14px;padding:18px 22px;margin-bottom:24px;
        box-shadow:0 4px 20px rgba(109,40,217,0.30);">
        <div style="font-size:30px;">{icon}</div>
        <div>
            <div style="font-size:20px;font-weight:700;color:#FFFFFF;letter-spacing:0.3px;">{title}</div>
            {sub_html}
        </div>
    </div>""", unsafe_allow_html=True)

def create_gauge(value, title, min_val=0, max_val=100, color="#8B5CF6"):
    fig = go.Figure(go.Indicator(
        mode="gauge+number", value=value,
        title={'text': title, 'font': {'color': "#6D28D9", 'size': 13}},
        number={'font': {'color': "#6D28D9", 'size': 18}},
        gauge={
            'axis': {'range': [min_val, max_val], 'tickwidth': 1, 'tickcolor': "#C4B5FD"},
            'bar': {'color': color},
            'bgcolor': "#F1F5F9",
            'borderwidth': 1,
            'bordercolor': "#C4B5FD",
            'steps': [{'range': [min_val, max_val], 'color': "#EDE9FE"}],
        }
    ))
    fig.update_layout(
        paper_bgcolor="#FFFFFF",
        font={'color': "#6B7280", 'family': "Inter"},
        height=230,
        margin=dict(l=10, r=10, t=40, b=10)
    )
    return fig

# ══════════════════════════════════════════════════════════
# AUTH PAGES
# ══════════════════════════════════════════════════════════
def signup_page():
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 1.4, 1])
    with col2:
        st.markdown("""<div style="text-align:center;margin-bottom:32px;">
            <div style="font-size:52px">🧠</div>
            <div style="font-size:26px;font-weight:700;color:#6D28D9;margin-top:8px;">Create Account</div>
            <div style="font-size:13px;color:#6B7280;margin-top:4px;">Join TextMorph</div>
        </div>""", unsafe_allow_html=True)
        with st.container():
            username = st.text_input("👤  Username")
            email    = st.text_input("📧  Email")
            password = st.text_input("🔑  Password", type="password")
            if password:
                strength, feedback = check_password_strength(password)
                color = {"Strong":"#8E4585","Medium":"#FACC15","Weak":"#ef4444"}[strength]
                st.markdown(f'<div style="margin:4px 0 8px;font-size:13px;">Strength: <span style="color:{color};font-weight:600;">{strength}</span></div>', unsafe_allow_html=True)
                if feedback: st.caption(f"💡 {', '.join(feedback)}")
            confirm_password = st.text_input("🔑  Confirm Password", type="password")
            question = st.selectbox("🛡️  Security Question", SECURITY_QUESTIONS)
            answer   = st.text_input("💬  Security Answer")
            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("✨  Create Account", type="primary"):
                username = username.strip(); email = email.strip().lower()
                answer_clean = answer.strip().lower()
                errors = []
                if not username: errors.append("Username required")
                if not is_valid_email(email): errors.append("Valid email required")
                if not password: errors.append("Password required")
                else:
                    s, fb = check_password_strength(password)
                    if s == "Weak": errors.append(f"Weak password: {', '.join(fb)}")
                if password != confirm_password: errors.append("Passwords don't match")
                if not answer_clean: errors.append("Security answer required")
                if errors:
                    for e in errors: st.error(e)
                    return
                if check_user_exists_by_email(email): st.error("Email already registered"); return
                if check_user_exists_by_username(username): st.error("Username taken"); return
                ok, msg = register_user(username, email, password, question, answer_clean)
                if ok: st.success("✅ Account created!"); time.sleep(1); st.session_state["page"]="login"; st.rerun()
                else: st.error(f"Error: {msg}")
            if st.button("← Back to Login", type="secondary"):
                st.session_state["page"]="login"; st.rerun()

def login_page():
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 1.4, 1])
    with col2:
        st.markdown("""<div style="text-align:center;margin-bottom:32px;">
            <div style="font-size:52px">TEXTMORPH</div>
            <div style="font-size:26px;font-weight:700;color:#6D28D9;margin-top:8px;">LLM LOGIN</div>
            <div style="font-size:13px;color:#6B7280;margin-top:4px;">Sign in to TextMorph</div>
        </div>""", unsafe_allow_html=True)
        email    = st.text_input("📧  Email")
        password = st.text_input("🔑  Password", type="password")
        st.markdown("<br>", unsafe_allow_html=True)
        if st.button("🚀  Sign In", type="primary", use_container_width=True):
            if not email or not password: st.error("All fields required"); return
            is_locked, wait_time = is_rate_limited(email)
            if is_locked: st.error(f"🔒 Account locked — try again in {wait_time}s"); return
            if email == ADMIN_EMAIL_ID and password == ADMIN_PASSWORD:
                token = create_access_token({"sub": email, "username": "admin"})
                st.session_state.update({"jwt_token":token,"username":"admin","nav_selected":"Admin"})
                st.success("✅ Admin access granted!"); time.sleep(1); st.rerun()
            success, uname = authenticate_user(email, password)
            if uname == "__LOCKED__" and not success:
                st.error("🔒 This account has been locked by an administrator."); return
            if success:
                token = create_access_token({"sub": email, "username": uname})
                st.session_state.update({"jwt_token":token,"username":uname})
                st.success(f"✅ Welcome back, {uname}!"); time.sleep(1); st.rerun()
            else:
                st.error("Invalid credentials")
                old_dt = check_is_old_password(email, password)
                if old_dt: st.warning(f"⚠️ This was a previously used password ({get_relative_time(old_dt)})")
                attempts, _ = get_login_attempts(email)
                remaining = MAX_LOGIN_ATTEMPTS - attempts
                if remaining > 0: st.caption(f"⚠️ {remaining} attempt(s) left before lockout")
        ca, cb = st.columns([1, 1])
        with ca:
            if st.button("📝  Register", type="secondary", use_container_width=True):
                st.session_state["page"]="signup"; st.rerun()
        with cb:
            if st.button("🔓  Forgot Password", type="secondary", use_container_width=True):
                st.session_state["page"]="forgot"; st.rerun()

def forgot_password_page():
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 1.4, 1])
    with col2:
        st.markdown("""<div style="text-align:center;margin-bottom:24px;">
            <div style="font-size:48px">🔓</div>
            <div style="font-size:24px;font-weight:700;color:#6D28D9;margin-top:8px;">Reset Password</div>
        </div>""", unsafe_allow_html=True)
        if "forgot_stage" not in st.session_state: st.session_state["forgot_stage"] = "email"
        stage = st.session_state["forgot_stage"]
        steps = ["📧 Email", "🛡️ Security", "📨 Send OTP", "✔️ Verify", "🔑 Reset"]
        idx   = ["email","security","otp_send","otp_verify","reset"].index(stage)
        st.markdown(f"""<div style="display:flex;justify-content:center;gap:8px;margin-bottom:24px;flex-wrap:wrap;">
            {"".join(f'<span style="background:{"rgba(139,92,246,0.15)" if i==idx else "#F1F5F9"};color:{"#6D28D9" if i==idx else "#6B7280"};padding:6px 14px;border-radius:20px;font-size:12px;font-weight:{"600" if i==idx else "400"};border:1px solid {"#C4B5FD" if i==idx else "transparent"};">{s}</span>' for i,s in enumerate(steps))}
        </div>""", unsafe_allow_html=True)
        if stage == "email":
            email_input = st.text_input("📧  Registered Email")
            if st.button("Next →", type="primary"):
                if not is_valid_email(email_input): st.error("Enter a valid email")
                elif not check_user_exists_by_email(email_input): st.error("Email not found")
                else:
                    st.session_state["reset_email"] = email_input.strip().lower()
                    st.session_state["forgot_stage"] = "security"; st.rerun()
        elif stage == "security":
            email  = st.session_state["reset_email"]
            result = get_security_question(email)
            if not result: st.error("Could not retrieve security question"); return
            question, correct_answer = result
            st.info(f"🛡️  {question}")
            user_answer = st.text_input("💬  Your Answer")
            if st.button("Verify →", type="primary"):
                if user_answer.strip().lower() == correct_answer:
                    st.session_state["forgot_stage"] = "otp_send"; st.rerun()
                else: st.error("Incorrect answer")
        elif stage == "otp_send":
            email = st.session_state["reset_email"]
            st.success(f"✅ Security verified!  OTP will go to: **{email}**")
            if st.button("📨  Send OTP", type="primary"):
                otp = generate_otp()
                ok, msg = send_otp_email(email, otp)
                if ok:
                    st.session_state["otp_token"] = create_otp_token(otp, email)
                    st.session_state["forgot_stage"] = "otp_verify"
                    st.success("OTP sent!"); time.sleep(1); st.rerun()
                else: st.error(f"Failed: {msg}")
        elif stage == "otp_verify":
            otp_input = st.text_input("🔢  6-digit OTP", max_chars=6)
            email     = st.session_state["reset_email"]
            ca, cb = st.columns(2)
            with ca:
                if st.button("Verify OTP →", type="primary"):
                    ok, msg = verify_otp_token(st.session_state.get("otp_token",""), otp_input, email)
                    if ok: st.session_state["forgot_stage"]="reset"; st.rerun()
                    else:  st.error(f"Failed: {msg}")
            with cb:
                if st.button("🔄 Resend", type="secondary"):
                    st.session_state["forgot_stage"]="otp_send"; st.rerun()
        elif stage == "reset":
            email = st.session_state["reset_email"]
            new_password = st.text_input("🔑  New Password", type="password")
            if new_password:
                strength, feedback = check_password_strength(new_password)
                color = {"Strong":"#8E4585","Medium":"#FACC15","Weak":"#ef4444"}[strength]
                st.markdown(f'<div style="margin:4px 0 8px;font-size:13px;">Strength: <span style="color:{color};font-weight:600;">{strength}</span></div>', unsafe_allow_html=True)
                if feedback: st.caption(f"💡 {', '.join(feedback)}")
            confirm_password = st.text_input("🔑  Confirm Password", type="password")
            if st.button("Update Password ✓", type="primary"):
                if new_password != confirm_password: st.error("Passwords don't match")
                else:
                    s, fb = check_password_strength(new_password)
                    if s == "Weak": st.error(f"Weak: {', '.join(fb)}")
                    elif check_password_reused(email, new_password): st.error("Cannot reuse previous password")
                    else:
                        update_password(email, new_password)
                        st.success("✅ Password updated!")
                        for k in ["forgot_stage","reset_email","otp_token"]: st.session_state.pop(k, None)
                        time.sleep(1); st.session_state["page"]="login"; st.rerun()
        st.markdown("---")
        if st.button("← Back to Login", type="secondary"):
            for k in ["forgot_stage","reset_email","otp_token"]: st.session_state.pop(k, None)
            st.session_state["page"]="login"; st.rerun()

# ══════════════════════════════════════════════════════════
# DASHBOARD PAGES
# ══════════════════════════════════════════════════════════
def readability_page():
    username = st.session_state.get("username","")
    page_header("📊", "Text Readability Analyzer", "Measure how easy your text is to understand")
    tab1, tab2 = st.tabs(["✍️  Type / Paste Text", "📂  Upload File"])
    text_input = ""
    with tab1:
        raw_text = st.text_area("Paste your text here (min 50 characters):", height=180,
                                label_visibility="collapsed",
                                placeholder="Paste your text here...")
        if raw_text: text_input = raw_text
    with tab2:
        try:
            uploaded_file = st.file_uploader("Upload a .txt or .pdf file", type=["txt","pdf"],
                                             label_visibility="collapsed")
            if uploaded_file:
                if uploaded_file.type == "application/pdf":
                    reader = PyPDF2.PdfReader(uploaded_file)
                    text_input = "".join([p.extract_text()+"\n" for p in reader.pages])
                    st.success(f"✅  Loaded {len(reader.pages)} page(s) from PDF")
                else:
                    text_input = uploaded_file.read().decode("utf-8")
                    st.success(f"✅  Loaded: {uploaded_file.name}")
        except Exception as e: st.error(f"Error: {e}")
    st.markdown("<br>", unsafe_allow_html=True)
    if st.button("🔍  Analyze Readability", type="primary"):
        if len(text_input) < 50: st.error("Text too short — minimum 50 characters"); return
        with st.spinner("Analyzing text..."):
            analyzer = ReadabilityAnalyzer(text_input)
            scores   = analyzer.get_all_metrics()
        avg_grade = (scores['Flesch-Kincaid Grade'] + scores['Gunning Fog'] +
                     scores['SMOG Index'] + scores['Coleman-Liau']) / 4
        if   avg_grade <= 6:  level, lcolor = "Beginner",     "#8E4585"
        elif avg_grade <= 10: level, lcolor = "Intermediate", "#8B5CF6"
        elif avg_grade <= 14: level, lcolor = "Advanced",     "#A78BFA"
        else:                 level, lcolor = "Expert",       "#6D28D9"
        st.markdown(f"""<div style="background:rgba(139,92,246,0.08);border:1px solid #C4B5FD;
            border-left:5px solid {lcolor};border-radius:12px;padding:20px 24px;margin:20px 0;
            display:flex;align-items:center;gap:20px;">
            <div style="font-size:40px;">📚</div>
            <div>
                <div style="font-size:20px;font-weight:700;color:{lcolor};">{level} Level</div>
                <div style="color:#6B7280;font-size:13px;margin-top:3px;">Approx. Grade {int(avg_grade)} · Flesch {round(scores['Flesch Reading Ease'],1)}</div>
            </div>
        </div>""", unsafe_allow_html=True)
        c1,c2,c3,c4,c5 = st.columns(5)
        for col, label, val in zip([c1,c2,c3,c4,c5],
            ["Sentences","Words","Syllables","Complex","Characters"],
            [analyzer.num_sentences,analyzer.num_words,analyzer.num_syllables,
             analyzer.complex_words,analyzer.char_count]):
            col.metric(label, val)
        st.markdown("<br>", unsafe_allow_html=True)
        gc1,gc2,gc3 = st.columns(3)
        gauge_data = [
            ("Flesch Reading Ease", 0, 100, "#8B5CF6", gc1),
            ("Flesch-Kincaid Grade", 0, 20, "#A78BFA", gc2),
            ("SMOG Index", 0, 20, "#6D28D9", gc3),
        ]
        gc4,gc5 = st.columns(2)
        gauge_data += [("Gunning Fog",0,20,"#8E4585",gc4), ("Coleman-Liau",0,20,"#C4B5FD",gc5)]
        for metric, mn, mx, clr, col in gauge_data:
            with col: st.plotly_chart(create_gauge(scores[metric],metric,mn,mx,clr), use_container_width=True)
        names  = list(scores.keys())
        vals   = list(scores.values())
        bar_fig = go.Figure(go.Bar(
            x=names, y=vals,
            marker_color=["#8B5CF6","#A78BFA","#8E4585","#6D28D9","#C4B5FD"],
            text=[f"{v:.1f}" for v in vals], textposition="outside",
            textfont=dict(color="#6B7280", size=12)
        ))
        bar_fig.update_layout(
            paper_bgcolor="#FFFFFF", plot_bgcolor="#F1F5F9",
            font=dict(color="#6B7280", family="Inter"),
            xaxis=dict(tickfont=dict(color="#6D28D9"), gridcolor="#C4B5FD"),
            yaxis=dict(tickfont=dict(color="#6D28D9"), gridcolor="#C4B5FD"),
            title=dict(text="Readability Score Overview", font=dict(color="#6D28D9",size=15), x=0.5),
            margin=dict(l=20,r=20,t=50,b=20), height=360
        )
        st.plotly_chart(bar_fig, use_container_width=True)
        st.dataframe(pd.DataFrame({
            "Metric": names,
            "Score":  [round(v,2) for v in vals],
            "Interpretation": [
                f"{'Easy' if scores['Flesch Reading Ease']>=60 else 'Challenging'} to read",
                f"Grade {int(scores['Flesch-Kincaid Grade'])} level",
                f"Grade {int(scores['SMOG Index'])} level",
                f"Grade {int(scores['Gunning Fog'])} level",
                f"Grade {int(scores['Coleman-Liau'])} level",
            ]
        }), use_container_width=True, hide_index=True)
        save_history_to_db(username,"Readability","textstat",
            f"Level:{level}|Grade:{int(avg_grade)}",text_input,
            f"Grade {int(avg_grade)} | {level} | Flesch={round(scores['Flesch Reading Ease'],1)}","English")

# ── FIX: Settings panel helper — renders a titled box WITHOUT creating phantom widgets ──
def _settings_header():
    """Renders the ⚙️ Settings title inside the CSS panel (no st widget overhead)."""
    st.markdown(
        '<div style="font-weight:600;font-size:14px;color:#6D28D9;margin-bottom:10px;">⚙️ Settings</div>',
        unsafe_allow_html=True
    )

def summarizer_page():
    username = st.session_state.get("username","")
    page_header("📝", "Text Summarizer", "Condense long text using state-of-the-art AI models")
    if 'summarization_history' not in st.session_state: st.session_state.summarization_history = []
    col1, col2 = st.columns([2, 1])
    with col1:
        st.markdown("**Input Text**")
        text_input = st.text_area("", height=200, key="sum_text",
                                  placeholder="Paste your text here (min 50 characters)...",
                                  label_visibility="collapsed")
        uploaded_file = st.file_uploader("Or upload a file", type=["txt","pdf"], key="sum_upload")
        if uploaded_file:
            if uploaded_file.type == "application/pdf":
                reader = PyPDF2.PdfReader(uploaded_file)
                text_input = "".join([p.extract_text()+"\n" for p in reader.pages])
            else: text_input = uploaded_file.read().decode("utf-8")
            st.success(f"✅  Loaded ({len(text_input.split())} words)")
    with col2:
        # ── FIX: use container with CSS class instead of raw markdown div ──
        with st.container():
            st.markdown('<div class="settings-panel-wrap">', unsafe_allow_html=True)
            _settings_header()
            summary_length = st.selectbox("📏  Length", ["Short","Medium","Long"])
            model_type     = st.selectbox("🤖  Model",  ["FLAN-T5","BART","Pegasus"])
            target_lang    = st.selectbox("🌐  Language", SUPPORTED_LANGUAGES)
            st.markdown('</div>', unsafe_allow_html=True)
        st.markdown("<br>", unsafe_allow_html=True)
        if st.button("▶  Generate Summary", type="primary", use_container_width=True):
            if len(text_input) < 50: st.error("Text too short (min 50 chars)")
            else:
                with st.spinner("Generating summary..."):
                    summary = engine.local_summarize(text_input, summary_length, model_type,
                                                     st.session_state.summarization_models, target_lang=target_lang)
                    if target_lang != "English":
                        with st.spinner(f"Translating to {target_lang}..."):
                            summary = engine.translate_text(summary, target_lang, st.session_state.translation_model)
                st.session_state.update({"last_summary":summary,"last_summary_text":text_input,"last_summary_lang":target_lang})
                st.session_state.summarization_history.append({
                    'timestamp': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    'input': text_input[:100]+("..." if len(text_input)>100 else ""),
                    'summary': summary, 'length': summary_length, 'model': model_type, 'lang': target_lang
                })
                save_history_to_db(username,"Summarize",model_type,
                    f"Length:{summary_length}|Lang:{target_lang}",text_input,summary,target_lang)
    if 'last_summary' in st.session_state:
        st.markdown("---")
        if st.session_state.get('last_summary_lang','English') != 'English':
            st.info(f"🌐  Output in **{st.session_state.last_summary_lang}**")
        r1, r2 = st.columns(2)
        with r1:
            st.markdown("**📄 Original**")
            st.markdown(f'<div class="result-box-original" style="white-space:pre-wrap;">{_html.escape(st.session_state.last_summary_text)}</div>', unsafe_allow_html=True)
            st.caption(f"📊 {len(st.session_state.last_summary_text.split())} words")
        with r2:
            st.markdown("**✨ Summary**")
            st.markdown(f'<div class="result-box-output" style="white-space:pre-wrap;">{_html.escape(st.session_state.last_summary)}</div>', unsafe_allow_html=True)
            st.caption(f"📊 {len(st.session_state.last_summary.split())} words")
        st.markdown("---")
        st.markdown("#### 💬 Rate this Summary")
        fb_col1, fb_col2 = st.columns([1, 2])
        with fb_col1:
            sum_rating = st.slider("⭐ Rating", 1, 5, 4, key="sum_fb_rating")
            stars = "⭐" * sum_rating + "☆" * (5 - sum_rating)
            st.markdown(f"<div style='font-size:20px;'>{stars}</div>", unsafe_allow_html=True)
        with fb_col2:
            sum_comment = st.text_input("💬 Comment (optional)", key="sum_fb_comment",
                                         placeholder="How was the summary quality?")
        if st.button("📤 Submit Feedback", type="secondary", key="sum_fb_submit"):
            save_feedback(username, sum_rating, "Summarize", sum_comment)
            st.success("✅ Thank you for your feedback!")
        with st.expander("📜  Session History"):
            for item in reversed(st.session_state.summarization_history[-5:]):
                st.markdown(f"**{item['timestamp']}** — `{item['length']}` via `{item['model']}`")
                st.info(item['input']); st.success(item['summary']); st.markdown("---")

def paraphraser_page():
    username = st.session_state.get("username","")
    page_header("🔄", "Paraphrase Engine", "Rewrite text with different words and styles using AI")
    if 'paraphrasing_history' not in st.session_state: st.session_state.paraphrasing_history = []
    col1, col2 = st.columns([2, 1])
    with col1:
        st.markdown("**Input Text**")
        text_input = st.text_area("", height=200, key="para_text",
                                  placeholder="Paste your text here (min 50 characters)...",
                                  label_visibility="collapsed")
        uploaded_file = st.file_uploader("Or upload a file", type=["txt","pdf"], key="para_upload")
        if uploaded_file:
            if uploaded_file.type == "application/pdf":
                reader = PyPDF2.PdfReader(uploaded_file)
                text_input = "".join([p.extract_text()+"\n" for p in reader.pages])
            else: text_input = uploaded_file.read().decode("utf-8")
            st.success(f"✅  Loaded ({len(text_input.split())} words)")
    with col2:
        # ── FIX: remove phantom st.markdown div wrapper, use container + CSS label ──
        with st.container():
            st.markdown('<div class="settings-panel-wrap">', unsafe_allow_html=True)
            _settings_header()
            complexity  = st.selectbox("🎚️  Complexity", ["Simple","Neutral","Advanced"])
            style       = st.selectbox("🎨  Style",      ["Simplification","Formalization","Creative"])
            model_type  = st.selectbox("🤖  Model",      ["FLAN-T5","BART"], key="para_model")
            target_lang = st.selectbox("🌐  Language",   SUPPORTED_LANGUAGES, key="para_lang")
            st.markdown('</div>', unsafe_allow_html=True)
        st.markdown("<br>", unsafe_allow_html=True)
        if st.button("▶  Generate Paraphrase", type="primary", use_container_width=True):
            if len(text_input) < 50: st.error("Text too short (min 50 chars)")
            else:
                with st.spinner("Generating paraphrase..."):
                    para = engine.paraphrase_with_model(text_input, complexity, style, model_type,
                                                        st.session_state.paraphrase_models, target_lang=target_lang)
                    if target_lang != "English":
                        with st.spinner(f"Translating to {target_lang}..."):
                            para = engine.translate_text(para, target_lang, st.session_state.translation_model)
                st.session_state.update({"last_para":para,"last_para_text":text_input,"last_para_lang":target_lang})
                st.session_state.paraphrasing_history.append({
                    'timestamp': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    'input': text_input[:100]+("..." if len(text_input)>100 else ""),
                    'paraphrase': para, 'complexity': complexity, 'style': style,
                    'model': model_type, 'lang': target_lang
                })
                save_history_to_db(username,"Paraphrase",model_type,
                    f"Complexity:{complexity}|Style:{style}|Lang:{target_lang}",text_input,para,target_lang)
    if 'last_para' in st.session_state:
        st.markdown("---")
        if st.session_state.get('last_para_lang','English') != 'English':
            st.info(f"🌐  Output in **{st.session_state.last_para_lang}**")
        r1, r2 = st.columns(2)
        with r1:
            st.markdown("**📄 Original**")
            st.markdown(f'<div class="result-box-original" style="white-space:pre-wrap;">{_html.escape(st.session_state.last_para_text)}</div>', unsafe_allow_html=True)
            st.caption(f"📊 {len(st.session_state.last_para_text.split())} words")
        with r2:
            st.markdown("**🔄 Paraphrased**")
            st.markdown(f'<div class="result-box-output" style="white-space:pre-wrap;">{_html.escape(st.session_state.last_para)}</div>', unsafe_allow_html=True)
            st.caption(f"📊 {len(st.session_state.last_para.split())} words")
        st.markdown("---")
        st.markdown("#### 💬 Rate this Paraphrase")
        fb_col1, fb_col2 = st.columns([1, 2])
        with fb_col1:
            para_rating = st.slider("⭐ Rating", 1, 5, 4, key="para_fb_rating")
            stars = "⭐" * para_rating + "☆" * (5 - para_rating)
            st.markdown(f"<div style='font-size:20px;'>{stars}</div>", unsafe_allow_html=True)
        with fb_col2:
            para_comment = st.text_input("💬 Comment (optional)", key="para_fb_comment",
                                          placeholder="How was the paraphrase quality?")
        if st.button("📤 Submit Feedback", type="secondary", key="para_fb_submit"):
            save_feedback(username, para_rating, "Paraphrase", para_comment)
            st.success("✅ Thank you for your feedback!")
        with st.expander("📜  Session History"):
            for item in reversed(st.session_state.paraphrasing_history[-5:]):
                st.markdown(f"**{item['timestamp']}** — `{item['complexity']}` `{item['style']}` `{item['model']}`")
                st.info(item['input']); st.success(item['paraphrase']); st.markdown("---")

def _simulate_training_metrics(model_arch, epochs, learning_rate, batch_size, dropout_rate, quantization):
    random.seed(hash(f"{model_arch}{epochs}{learning_rate}{batch_size}{dropout_rate}{quantization}"))
    lr_val       = float(learning_rate)
    base_loss    = {"T5-Small":0.55,"BART-Base":0.48,"FLAN-T5":0.42}.get(model_arch, 0.50)
    epoch_factor = 1.0 - (min(epochs,10)*0.06)
    lr_factor    = 1.0 - (lr_val*8000)
    dropout_bonus= dropout_rate*0.08
    quant_penalty= {"FP16 (None)":0.0,"8-bit":0.02,"4-bit":0.05}.get(quantization, 0.0)
    final_loss   = round(max(0.15, base_loss*epoch_factor*lr_factor+dropout_bonus+quant_penalty+random.uniform(-0.03,0.03)),2)
    accuracy     = round(min(95, 65+(epochs*2.5)+(1-final_loss)*20+random.uniform(-2,3)),1)
    rouge_l      = round(random.uniform(1.5,4.0)+epochs*0.15,1)
    bleu         = round(0.25+(epochs*0.02)+(1-final_loss)*0.15+random.uniform(-0.03,0.03),2)
    loss_curve   = []
    curr = base_loss+1.0
    for _ in range(epochs):
        curr = curr*(0.6+random.uniform(-0.05,0.05))+random.uniform(-0.02,0.02)
        loss_curve.append(round(max(final_loss,curr),3))
    loss_curve[-1] = final_loss
    return {"final_loss":str(final_loss),"delta_loss":str(round(random.uniform(-0.08,-0.15),2)),
            "accuracy":f"{accuracy}%","delta_acc":f"+{round(random.uniform(1,6),1)}%",
            "rouge_l":f"+{rouge_l}","delta_rouge":f"+{round(random.uniform(0.3,1.2),1)}",
            "bleu":str(bleu),"delta_bleu":f"+{round(random.uniform(0.02,0.08),2)}",
            "loss_curve":loss_curve,"epochs_x":list(range(1,epochs+1))}

def augmentation_page():
    page_header("🗃️", "Dataset Augmentation & Fine-Tuning", "Build datasets and fine-tune models on custom data")
    tab_explore, tab_tune, tab_studio = st.tabs(["📊  Dataset Explorer","🛠️  Model Tuning","🧪  Augmentation Studio"])
    with tab_explore:
        datasets = {"CNN/DailyMail":{"samples":311029,"type":"News Summarization","avg_words":781},
                    "XSum":         {"samples":226711,"type":"Extreme Summarization","avg_words":431},
                    "PAWS":         {"samples":108461,"type":"Paraphrase","avg_words":21}}
        selected = st.selectbox("📂  Select Dataset", list(datasets.keys()))
        c1,c2,c3 = st.columns(3)
        c1.metric("Total Samples",  f"{datasets[selected]['samples']:,}")
        c2.metric("Task Type",       datasets[selected]['type'])
        c3.metric("Avg Doc Length", f"{datasets[selected]['avg_words']} words")
        st.markdown("**🧹 Cleaning Filters**")
        col_a, col_b = st.columns(2)
        with col_a: min_len = st.slider("Min Words",5,100,10)
        with col_b: max_len = st.slider("Max Words",100,2000,1000)
        filtered = int(datasets[selected]['samples']*(0.9-(min_len/1000)-(1000-max_len)/2000))
        st.success(f"✅  Filtered size: **{filtered:,} pairs**")
        mock_rows = []
        for i in range(1, 6):
            mock_rows.append({
                "S.No":             i,
                "ID":            f"{selected[:3]}-00{i}",
                "Original Text": f"This is sample document {i} sourced from the {selected} dataset, containing representative content for NLP tasks.",
                "Target Output": f"Processed and cleaned target output {i} ready for model training.",
                "Word Count":    [140, 432, 21, 89, 210][i-1],
                "Complexity":    [8.4, 12.1, 4.2, 7.9, 9.3][i-1],
                "Status":        "✅ Clean"
            })
        mock_df = pd.DataFrame(mock_rows)
        st.dataframe(mock_df, use_container_width=True, hide_index=True, height=280)
        st.markdown("---")
        st.markdown("**🗂️ Card Preview**")
        for row in mock_rows[:3]:
            with st.container(border=True):
                cc1, cc2, cc3 = st.columns([1, 3, 3])
                with cc1:
                    st.markdown(f"<div style='text-align:center;font-size:22px;font-weight:700;color:#6D28D9;'>#{row['S.No']}</div>", unsafe_allow_html=True)
                    st.caption(row["ID"])
                    st.markdown(f"<div style='font-size:11px;color:#8B5CF6;'>📝 {row['Word Count']} words</div>", unsafe_allow_html=True)
                with cc2:
                    st.caption("**📄 Original Text**")
                    st.info(row["Original Text"])
                with cc3:
                    st.caption("**✨ Target Output**")
                    st.success(row["Target Output"])
    with tab_tune:
        c1,c2,c3 = st.columns(3)
        with c1:
            model_arch = st.selectbox("🏗️  Architecture", ["T5-Small","BART-Base","FLAN-T5"])
            epochs     = st.slider("🔁  Epochs",1,10,3)
        with c2:
            quantization = st.selectbox("⚡  Quantization", ["FP16 (None)","8-bit","4-bit"])
            batch_size   = st.slider("📦  Batch Size",8,32,16)
        with c3:
            learning_rate = st.selectbox("📉  Learning Rate", ["1e-5","2e-5","3e-5"])
            dropout_rate  = st.slider("💧  Dropout",0.0,0.5,0.1)
        if st.button("🚀  Start Fine-Tuning", type="primary", use_container_width=True):
            with st.spinner(f"Fine-tuning {model_arch}..."):
                bar = st.progress(0)
                for i in range(100): time.sleep(0.01); bar.progress(i+1)
                st.success(f"✅  {model_arch} fine-tuned successfully!")
                metrics = _simulate_training_metrics(model_arch,epochs,learning_rate,batch_size,dropout_rate,quantization)
                m1,m2,m3,m4 = st.columns(4)
                m1.metric("Final Loss",metrics["final_loss"],metrics["delta_loss"])
                m2.metric("Train Acc.",metrics["accuracy"],metrics["delta_acc"])
                m3.metric("ROUGE-L",  metrics["rouge_l"],  metrics["delta_rouge"])
                m4.metric("BLEU",     metrics["bleu"],     metrics["delta_bleu"])
                fig = go.Figure(go.Scatter(x=metrics["epochs_x"],y=metrics["loss_curve"],
                    mode='lines+markers',
                    line=dict(color='#8B5CF6',width=3),
                    marker=dict(size=8,color='#8B5CF6',line=dict(color='#C4B5FD',width=1.5))))
                fig.update_layout(
                    title=f"Training Loss — {model_arch}",
                    xaxis_title="Epoch", yaxis_title="Loss",
                    paper_bgcolor="#FFFFFF", plot_bgcolor="#F1F5F9",
                    font=dict(color="#6B7280",family="Inter"),
                    height=300, margin=dict(l=0,r=0,t=40,b=0))
                st.plotly_chart(fig, use_container_width=True)
    with tab_studio:
        aug_input = st.text_area("Paragraphs (separated by blank lines):", height=180, key="aug_text",
            value="The quick brown fox jumps over the lazy dog.\n\nAI is transforming the modern world rapidly.")
        col_a, col_b = st.columns(2)
        with col_a: aug_type = st.selectbox("🔧  Transform", ["Paraphrasing","Summarization"])
        with col_b:
            aug_setting = st.selectbox("⚙️  Setting",
                ["Short","Medium","Long"] if aug_type=="Summarization" else ["Advanced","Simple","Neutral"])
        if st.button("🚀  Generate Dataset", type="secondary", use_container_width=True):
            paras = [p.strip() for p in aug_input.split('\n\n') if len(p.strip())>10]
            if not paras: st.error("Enter at least one paragraph.")
            else:
                results = []; bar = st.progress(0)
                for idx, para in enumerate(paras):
                    res = engine.local_summarize(para,aug_setting,"BART",st.session_state.summarization_models) \
                          if aug_type=="Summarization" \
                          else engine.paraphrase_with_model(para,aug_setting,"Creative","FLAN-T5",st.session_state.paraphrase_models)
                    results.append({"S.No":idx+1,"Original":para[:200],"Generated":res[:200],
                                    "Orig WC":len(para.split()),"Gen WC":len(res.split()),
                                    "Delta":f"{len(res.split())-len(para.split()):+d}"})
                    bar.progress((idx+1)/len(paras))
                st.success(f"✅  Generated {len(results)} pairs!")
                results_df = pd.DataFrame(results)
                st.dataframe(results_df, use_container_width=True, hide_index=True, height=300)
                st.markdown("---")
                st.markdown("**🗂️ Card View**")
                for row in results:
                    with st.container(border=True):
                        hc1, hc2 = st.columns([1, 8])
                        with hc1:
                            st.markdown(f"<div style='text-align:center;font-size:24px;font-weight:700;color:#6D28D9;padding-top:6px;'>#{row['S.No']}</div>", unsafe_allow_html=True)
                        with hc2:
                            bc1, bc2 = st.columns(2)
                            with bc1:
                                st.caption("**📄 Original**")
                                st.info(row["Original"])
                            with bc2:
                                st.caption("**✨ Generated**")
                                st.success(row["Generated"])
                st.download_button("📥  Download CSV",
                    data=pd.DataFrame(results).to_csv(index=False).encode('utf-8'),
                    file_name="augmented_dataset.csv", mime="text/csv", use_container_width=True)

# ══════════════════════════════════════════════════════════
# HISTORY PAGE
# ══════════════════════════════════════════════════════════
def history_page(username, is_admin=False):
    page_header("🕘", "Activity History", "Complete log of all your AI operations")

    raw  = get_all_history_from_db(300) if is_admin else get_history_from_db(username, 200)
    if is_admin:
        rows = [{"_username":r[0],"action_type":r[1],"model_used":r[2],"settings":r[3],
                 "input_snippet":r[4],"output_snippet":r[5],"language":r[6],
                 "wc_in":r[7],"wc_out":r[8],"cc_in":r[9],"cc_out":r[10],"created_at":r[11]} for r in raw]
    else:
        rows = [{"_id":r[0],"action_type":r[1],"model_used":r[2],"settings":r[3],
                 "input_snippet":r[4],"output_snippet":r[5],"language":r[6],
                 "wc_in":r[7],"wc_out":r[8],"cc_in":r[9],"cc_out":r[10],"created_at":r[11]} for r in raw]

    def reading_time(wc):
        mins = max(1, round(wc / 200))
        return f"{mins} min read"

    def compression_ratio(wc_in, wc_out):
        if not wc_in or wc_in == 0: return "—"
        ratio = round((1 - wc_out / wc_in) * 100, 1)
        return f"{ratio}% ↓" if ratio > 0 else f"{abs(ratio)}% ↑"

    def fmt_dt(dt):
        if not dt: return "N/A"
        try:
            IST_OFFSET = datetime.timedelta(hours=5, minutes=30)
            if isinstance(dt, str):
                dt = datetime.datetime.strptime(dt[:19], "%Y-%m-%d %H:%M:%S")
            if hasattr(dt, "tzinfo") and dt.tzinfo is not None:
                dt = dt.replace(tzinfo=None)
            return (dt + IST_OFFSET).strftime("%d %b %Y  %I:%M:%S %p IST")
        except: return str(dt)

    total    = len(rows)
    n_sum    = sum(1 for r in rows if r["action_type"] == "Summarize")
    n_para   = sum(1 for r in rows if r["action_type"] == "Paraphrase")
    n_read   = sum(1 for r in rows if r["action_type"] == "Readability")
    total_wc = sum((r["wc_in"] or 0) for r in rows)
    top_model = "—"
    if rows:
        from collections import Counter
        mc = Counter(r["model_used"] for r in rows if r["model_used"])
        top_model = mc.most_common(1)[0][0] if mc else "—"

    s1,s2,s3,s4,s5,s6 = st.columns(6)
    for col, num, label, icon in [
        (s1, total,           "Total Actions",   "📋"),
        (s2, n_sum,           "Summaries",       "📝"),
        (s3, n_para,          "Paraphrases",     "🔄"),
        (s4, n_read,          "Readability",     "📊"),
        (s5, f"{total_wc:,}", "Words Processed", "📖"),
        (s6, top_model,       "Top Model",       "🤖"),
    ]:
        col.markdown(f"""<div class="stat-card">
            <div style="font-size:22px;margin-bottom:4px;">{icon}</div>
            <div class="stat-num">{num}</div>
            <div class="stat-label">{label}</div>
        </div>""", unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)
    tab_table, tab_cards, tab_analytics = st.tabs(["📋  Table View", "🗂️  Card View", "📈  Analytics"])

    with tab_table:
        _render_filters_and_table(rows, username, is_admin, fmt_dt, reading_time, compression_ratio)
    with tab_cards:
        _render_card_view(rows, username, is_admin, fmt_dt, reading_time, compression_ratio)
    with tab_analytics:
        _render_analytics(rows)


def _render_filters_and_table(rows, username, is_admin, fmt_dt, reading_time, compression_ratio):
    cf1,cf2,cf3,cf4 = st.columns([2,2,2,1])
    with cf1: search_q    = st.text_input("🔍 Search", placeholder="keyword...", key="tbl_search")
    with cf2: filter_type = st.selectbox("Type", ["All","Summarize","Paraphrase","Readability"], key="tbl_type")
    with cf3: filter_model= st.selectbox("Model",
                ["All"]+sorted({r["model_used"] for r in rows if r["model_used"]}), key="tbl_model")
    with cf4: per_page    = st.selectbox("Per page",[10,25,50], key="tbl_pp")

    if not is_admin:
        if st.button("🗑️  Clear My History", type="secondary", key="clear_tbl"):
            clear_user_history(username); st.success("Cleared."); time.sleep(0.4); st.rerun()

    filtered = _apply_filters(rows, filter_type, filter_model, search_q)
    if not filtered:
        st.info("📭  No records match your filters."); return

    total_pages = max(1, (len(filtered) + per_page - 1) // per_page)
    if "tbl_page" not in st.session_state: st.session_state["tbl_page"] = 1
    st.session_state["tbl_page"] = min(st.session_state["tbl_page"], total_pages)
    page_num  = st.session_state["tbl_page"]
    start_i   = (page_num - 1) * per_page
    page_rows = filtered[start_i : start_i + per_page]

    st.markdown(
        f'<div style="color:#6B7280;font-size:12px;margin-bottom:8px;">'
        f'Showing {start_i+1}–{min(start_i+per_page,len(filtered))} of {len(filtered)} '
        f'&nbsp;·&nbsp; Page {page_num}/{total_pages} '
        f'&nbsp;·&nbsp; <span style="color:#A78BFA;">Click any row to expand full details</span></div>',
        unsafe_allow_html=True)

    icon_map = {"Summarize":"📝","Paraphrase":"🔄","Readability":"📊"}
    table_rows = []
    for idx_r, r in enumerate(page_rows):
        wc_in  = r.get("wc_in",0)  or 0
        wc_out = r.get("wc_out",0) or 0
        row = {
            "S.No":      start_i + idx_r + 1,
            "Date":      fmt_dt(r.get("created_at")),
            "Type":      icon_map.get(r["action_type"],"") + " " + r["action_type"],
            "Model":     (r.get("model_used") or "—"),
            "Lang":      (r.get("language") or "EN"),
            "W.In":      wc_in,
            "W.Out":     wc_out,
            "Compress":  compression_ratio(wc_in, wc_out),
            "Input ↓":   (r.get("input_snippet")  or "")[:45] + ("…" if len(r.get("input_snippet") or "") > 45 else ""),
            "Output ↓":  (r.get("output_snippet") or "")[:45] + ("…" if len(r.get("output_snippet") or "") > 45 else ""),
        }
        if is_admin and "_username" in r: row["User"] = r["_username"]
        table_rows.append(row)

    df = pd.DataFrame(table_rows)

    col_cfg = {
        "S.No":     st.column_config.NumberColumn("S.No",    width=40),
        "Date":     st.column_config.TextColumn("🕐 Date",   width=160),
        "Type":     st.column_config.TextColumn("Type",      width=120),
        "Model":    st.column_config.TextColumn("Model",     width=90),
        "Lang":     st.column_config.TextColumn("Lang",      width=70),
        "W.In":     st.column_config.NumberColumn("W.In",    width=55, format="%d"),
        "W.Out":    st.column_config.NumberColumn("W.Out",   width=60, format="%d"),
        "Compress": st.column_config.TextColumn("Compress",  width=80),
        "Input ↓":  st.column_config.TextColumn("Input ↓",  width=200),
        "Output ↓": st.column_config.TextColumn("Output ↓", width=200),
    }
    if is_admin: col_cfg["User"] = st.column_config.TextColumn("User", width=90)

    event = st.dataframe(
        df, use_container_width=True, hide_index=True,
        column_config=col_cfg, height=400,
        on_select="rerun", selection_mode="single-row",
        key="hist_table_sel",
    )

    selected_rows = event.selection.rows if hasattr(event, "selection") else []
    if selected_rows:
        sel_idx = selected_rows[0]
        r = page_rows[sel_idx]
        wc_in  = r.get("wc_in",0)  or 0
        wc_out = r.get("wc_out",0) or 0
        atype  = r["action_type"]
        inp_lbl = "Question / Input" if atype in ("Summarize","Paraphrase") else "Analyzed Text"
        out_lbl = {"Summarize":"Summary Result","Paraphrase":"Paraphrased Result","Readability":"Analysis Result"}.get(atype,"Output")
        _rm = st.columns(4)
        _rm[0].metric("Operation", atype)
        _rm[1].metric("Model", r.get("model_used") or "—")
        _rm[2].metric("Language", r.get("language") or "English")
        _rm[3].metric("Compression", compression_ratio(wc_in, wc_out))
        ec1, ec2 = st.columns(2)
        with ec1:
            st.markdown(f"**📄 {inp_lbl}**")
            _inp_val = r.get("input_snippet") or ""
            if _inp_val: st.text_area("", value=_inp_val, height=180, disabled=True, key=f"tbl_inp_{sel_idx}")
            else: st.caption("No input stored")
        with ec2:
            st.markdown(f"**✨ {out_lbl}**")
            _out_val = r.get("output_snippet") or ""
            if _out_val: st.text_area("", value=_out_val, height=180, disabled=True, key=f"tbl_out_{sel_idx}")
            else: st.caption("No output stored")
    else:
        st.markdown(
            '<div style="text-align:center;color:#A78BFA;font-size:12px;padding:8px;'
            'border:1px dashed #C4B5FD;border-radius:8px;margin-top:8px;">'
            '👆 Click any row above to see full input & output details</div>',
            unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)
    pc1,pc2,pc3,pc4,pc5 = st.columns([1,1,2,1,1])
    with pc1:
        if st.button("⏮️", use_container_width=True, key="pg_first"):
            st.session_state["tbl_page"]=1; st.rerun()
    with pc2:
        if st.button("◀", use_container_width=True, key="pg_prev"):
            st.session_state["tbl_page"]=max(1,page_num-1); st.rerun()
    with pc3:
        st.markdown(
            f'<div style="text-align:center;padding:8px;color:#6B7280;font-size:13px;">'
            f'Page <b style="color:#6D28D9">{page_num}</b> / <b style="color:#6D28D9">{total_pages}</b></div>',
            unsafe_allow_html=True)
    with pc4:
        if st.button("▶", use_container_width=True, key="pg_next"):
            st.session_state["tbl_page"]=min(total_pages,page_num+1); st.rerun()
    with pc5:
        if st.button("⏭️", use_container_width=True, key="pg_last"):
            st.session_state["tbl_page"]=total_pages; st.rerun()

    ist_now = (datetime.datetime.utcnow()+datetime.timedelta(hours=5,minutes=30)).strftime("%Y%m%d_%H%M")
    icon_map2 = {"Summarize":"📝","Paraphrase":"🔄","Readability":"📊"}
    full_df = _build_full_df(filtered, is_admin, fmt_dt, reading_time, compression_ratio, icon_map2)
    st.download_button("📥  Export All as CSV",
        data=full_df.to_csv(index=False).encode("utf-8"),
        file_name=f"history_{username}_{ist_now}.csv",
        mime="text/csv", use_container_width=True)


def _render_card_view(rows, username, is_admin, fmt_dt, reading_time, compression_ratio):
    cf1,cf2,cf3 = st.columns([2,2,2])
    with cf1: search_q    = st.text_input("🔍 Search", placeholder="keyword...", key="card_search")
    with cf2: filter_type = st.selectbox("Type",["All","Summarize","Paraphrase","Readability"],key="card_type")
    with cf3: filter_model= st.selectbox("Model",
                ["All"]+sorted({r["model_used"] for r in rows if r["model_used"]}),key="card_model")

    if not is_admin:
        if st.button("🗑️  Clear My History", type="secondary", key="clear_card"):
            clear_user_history(username); st.success("Cleared."); time.sleep(0.4); st.rerun()

    filtered = _apply_filters(rows, filter_type, filter_model, search_q)
    if not filtered:
        st.markdown("""<div style="text-align:center;padding:48px;color:#A78BFA;">
            <div style="font-size:44px;">📭</div>
            <div style="margin-top:10px;font-size:15px;">No records found</div>
        </div>""", unsafe_allow_html=True); return

    st.markdown(f'<div style="color:#6B7280;font-size:13px;margin-bottom:12px;">{len(filtered)} records</div>', unsafe_allow_html=True)

    for r in filtered[:50]:
        atype  = r["action_type"]
        model  = r.get("model_used") or "N/A"
        lang   = r.get("language")   or "English"
        sett   = r.get("settings")   or ""
        inp    = r.get("input_snippet")  or ""
        out    = r.get("output_snippet") or ""
        wc_in  = r.get("wc_in", 0)  or 0
        wc_out = r.get("wc_out", 0) or 0
        ts     = fmt_dt(r.get("created_at"))
        cr     = compression_ratio(wc_in, wc_out)
        rt     = reading_time(wc_out)
        uid    = r.get("_id", id(r))
        _type_emoji = {"Summarize":"📝","Paraphrase":"🔄","Readability":"📊"}.get(atype,"📋")
        _admin_tag = f"  |  {r['_username']}" if is_admin and "_username" in r else ""

        with st.container(border=True):
            _c1, _c2 = st.columns([3, 2])
            with _c1: st.write(f"{_type_emoji} **{atype}**  —  {model}  ·  {lang}")
            with _c2: st.caption(f"{cr}  ·  {rt}  ·  {ts}{_admin_tag}")
            _m1, _m2, _m3 = st.columns(3)
            _m1.caption(f"**Model:** {model}")
            _m2.caption(f"**Lang:** {lang}")
            _m3.caption(f"**Words:** {wc_in} → {wc_out}")
            if inp: st.info(inp[:250] + ("…" if len(inp) > 250 else ""))
            if out: st.success(out[:250] + ("…" if len(out) > 250 else ""))

    if len(filtered) > 50:
        st.info(f"Showing first 50 of {len(filtered)} cards. Use Table View for full list.")


def _render_analytics(rows):
    if not rows:
        st.info("📭  No data yet to analyze."); return

    from collections import Counter

    a1, a2 = st.columns(2)
    with a1:
        st.markdown("#### Action Type Distribution")
        type_counts = Counter(r["action_type"] for r in rows)
        fig_pie = go.Figure(go.Pie(
            labels=list(type_counts.keys()),
            values=list(type_counts.values()),
            marker=dict(colors=["#8B5CF6","#8E4585","#A78BFA"], line=dict(color="#FFFFFF",width=2)),
            hole=0.45,
        ))
        fig_pie.update_layout(paper_bgcolor="#FFFFFF", font=dict(color="#6B7280",family="Inter"), height=280, margin=dict(l=10,r=10,t=30,b=10))
        st.plotly_chart(fig_pie, use_container_width=True)

    with a2:
        st.markdown("#### Model Usage")
        model_counts = Counter(r["model_used"] for r in rows if r["model_used"])
        fig_bar = go.Figure(go.Bar(
            x=list(model_counts.values()), y=list(model_counts.keys()), orientation="h",
            marker=dict(color=["#8B5CF6","#8E4585","#A78BFA","#6D28D9","#C4B5FD"][:len(model_counts)]),
            text=list(model_counts.values()), textposition="auto"
        ))
        fig_bar.update_layout(paper_bgcolor="#FFFFFF", plot_bgcolor="#F1F5F9", font=dict(color="#6B7280",family="Inter"), height=280, margin=dict(l=10,r=10,t=30,b=10))
        st.plotly_chart(fig_bar, use_container_width=True)


def _apply_filters(rows, filter_type, filter_model, search_q):
    filtered = rows
    if filter_type  != "All": filtered = [r for r in filtered if r["action_type"]==filter_type]
    if filter_model != "All": filtered = [r for r in filtered if r.get("model_used")==filter_model]
    if search_q:
        q = search_q.lower()
        filtered = [r for r in filtered if q in (r.get("input_snippet") or "").lower()
                    or q in (r.get("output_snippet") or "").lower()]
    return filtered


def _build_full_df(filtered, is_admin, fmt_dt, reading_time, compression_ratio, icon_map):
    table_rows = []
    for r in filtered:
        wc_in  = r.get("wc_in",0)  or 0
        wc_out = r.get("wc_out",0) or 0
        row = {
            "Date & Time (IST)": fmt_dt(r.get("created_at")),
            "Type":              icon_map.get(r["action_type"],"") + " " + r["action_type"],
            "Model":             r.get("model_used") or "—",
            "Settings":          r.get("settings")   or "—",
            "Language":          r.get("language")   or "English",
            "Words In":          wc_in,
            "Words Out":         wc_out,
            "Compression":       compression_ratio(wc_in, wc_out),
            "Read Time":         reading_time(wc_out),
            "Input":             r.get("input_snippet")  or "",
            "Output":            r.get("output_snippet") or "",
        }
        if is_admin and "_username" in r: row["User"] = r["_username"]
        table_rows.append(row)
    return pd.DataFrame(table_rows)


# ══════════════════════════════════════════════════════════
# CHAT PAGE
# ══════════════════════════════════════════════════════════
def chat_page():
    payload = verify_token(st.session_state["jwt_token"])
    if not payload:
        st.session_state.update({"jwt_token": None, "page": "login"}); st.rerun(); return
    username = payload.get("username", "User")

    page_header("💬", "AI Chat", "Have a conversation with the AI assistant")

    if "chat_messages" not in st.session_state:
        history = get_chat_history(username)
        st.session_state["chat_messages"] = [
            {"role": r[0], "content": r[1], "model": r[2]} for r in history
        ]
    if "chat_model" not in st.session_state: st.session_state["chat_model"] = "FLAN-T5"
    if "chat_lang"  not in st.session_state: st.session_state["chat_lang"]  = "English"

    col_m, col_l, col_clr = st.columns([2, 2, 1])
    with col_m:
        model_choice = st.selectbox("🤖 Model", ["FLAN-T5","BART","Pegasus"],
            index=["FLAN-T5","BART","Pegasus"].index(st.session_state["chat_model"]), key="chat_model_sel")
        st.session_state["chat_model"] = model_choice
    with col_l:
        lang_choice = st.selectbox("🌐 Language", SUPPORTED_LANGUAGES,
            index=SUPPORTED_LANGUAGES.index(st.session_state["chat_lang"]), key="chat_lang_sel")
        st.session_state["chat_lang"] = lang_choice
    with col_clr:
        st.markdown("<br>", unsafe_allow_html=True)
        if st.button("🗑️ Clear Chat", use_container_width=True, type="secondary"):
            clear_chat_history(username); st.session_state["chat_messages"] = []; st.rerun()

    st.markdown("<hr style='border-color:#C4B5FD;margin:8px 0 16px;'>", unsafe_allow_html=True)

    if not st.session_state["chat_messages"]:
        st.markdown("""<div style="text-align:center;padding:60px 20px;color:#A78BFA;">
            <div style="font-size:48px;margin-bottom:12px;">💬</div>
            <div style="font-size:16px;font-weight:600;color:#6D28D9;">Start a conversation</div>
            <div style="font-size:13px;margin-top:6px;color:#6B7280;">Type a message below to begin</div>
        </div>""", unsafe_allow_html=True)
    else:
        for msg in st.session_state["chat_messages"]:
            role = msg["role"]; content = msg["content"]; mdl = msg.get("model","")
            if role == "user":
                st.markdown(f"""<div style="display:flex;justify-content:flex-end;margin:8px 0;">
                    <div style="background:linear-gradient(135deg,#8B5CF6,#A78BFA);color:#fff;
                        border-radius:18px 18px 4px 18px;padding:12px 18px;max-width:70%;font-size:14px;">
                        {content}</div>
                    <div style="font-size:24px;margin-left:10px;align-self:flex-end;">👤</div>
                </div>""", unsafe_allow_html=True)
            else:
                st.markdown(f"""<div style="display:flex;justify-content:flex-start;margin:8px 0;">
                    <div style="font-size:24px;margin-right:10px;align-self:flex-end;">🤖</div>
                    <div style="background:#FFFFFF;border:1px solid #C4B5FD;color:#1F2937;
                        border-radius:18px 18px 18px 4px;padding:12px 18px;max-width:70%;font-size:14px;">
                        {content}</div>
                </div>""", unsafe_allow_html=True)

    inp_col, btn_col = st.columns([5, 1])
    with inp_col:
        user_input = st.text_input("Message", placeholder="Type your message…",
                                   label_visibility="collapsed", key="chat_input")
    with btn_col:
        send = st.button("Send 🚀", type="primary", use_container_width=True)

    if send and user_input.strip():
        user_text = user_input.strip()
        st.session_state["chat_messages"].append({"role":"user","content":user_text,"model":model_choice})
        save_chat_message(username, "user", user_text, model_choice)
        with st.spinner("🤖 Thinking…"):
            try:
                prompt = f"Answer this question helpfully and concisely: {user_text}"
                models_dict = st.session_state.get("paraphrase_models", {})
                summ_dict   = st.session_state.get("summarization_models", {})
                translation_model = st.session_state.get("translation_model")
                ai_reply = None
                if model_choice == "FLAN-T5" and models_dict.get("flan_t5"):
                    ai_reply = engine.paraphrase_with_model(prompt,"Neutral","Formal","FLAN-T5",models_dict)
                elif model_choice == "BART" and summ_dict.get("bart"):
                    ai_reply = engine.local_summarize(prompt,"Medium","BART",summ_dict)
                elif model_choice == "Pegasus" and summ_dict.get("pegasus"):
                    ai_reply = engine.local_summarize(prompt,"Medium","Pegasus",summ_dict)
                if not ai_reply or len(ai_reply.strip()) < 5:
                    ai_reply = f"I received your message: \"{user_text}\". The {model_choice} model is processing."
                if lang_choice != "English" and translation_model:
                    ai_reply = engine.translate_text(ai_reply, lang_choice, translation_model)
            except Exception as e:
                ai_reply = f"⚠️ Error: {str(e)}"
        st.session_state["chat_messages"].append({"role":"assistant","content":ai_reply,"model":model_choice})
        save_chat_message(username, "assistant", ai_reply, model_choice)
        st.rerun()


# ══════════════════════════════════════════════════════════
# PROFILE PAGE
# ══════════════════════════════════════════════════════════
def profile_page():
    payload = verify_token(st.session_state["jwt_token"])
    if not payload: st.session_state.update({"jwt_token":None,"page":"login"}); st.rerun(); return
    username = payload.get("username","User")
    email    = payload.get("sub","")

    page_header("👤", "My Profile", "Manage your account settings and preferences")

    tab1, tab2, tab3, tab4 = st.tabs(["🖼️  Avatar", "📧  Update Email", "🔑  Change Password", "💬  Feedback"])

    with tab1:
        st.markdown("#### Profile Picture")
        current_avatar = get_user_avatar(email)
        col_av, col_up = st.columns([1, 2])
        with col_av:
            if current_avatar:
                st.markdown(f'<img src="data:image/png;base64,{current_avatar}" style="width:120px;height:120px;border-radius:50%;border:3px solid #8B5CF6;object-fit:cover;display:block;margin:auto;" />', unsafe_allow_html=True)
            else:
                initials = username[0].upper()
                st.markdown(f'<div style="width:120px;height:120px;border-radius:50%;background:linear-gradient(135deg,#8B5CF6,#A78BFA);display:flex;align-items:center;justify-content:center;font-size:48px;font-weight:700;color:#fff;margin:auto;">{initials}</div>', unsafe_allow_html=True)
        with col_up:
            st.markdown("<br>", unsafe_allow_html=True)
            uploaded = st.file_uploader("Upload new avatar", type=["png","jpg","jpeg","gif","webp"], key="avatar_upload")
            if uploaded:
                img_bytes = uploaded.read()
                b64 = base64.b64encode(img_bytes).decode()
                st.markdown(f'<img src="data:image/png;base64,{b64}" style="width:80px;height:80px;border-radius:50%;object-fit:cover;border:2px dashed #A78BFA;display:block;margin:4px 0 10px;" />', unsafe_allow_html=True)
                if st.button("💾  Save Avatar", type="primary", key="save_av"):
                    update_user_avatar(email, b64)
                    st.success("✅ Avatar updated!"); time.sleep(0.5); st.rerun()
            if current_avatar:
                if st.button("🗑️  Remove Avatar", type="secondary", key="del_av"):
                    update_user_avatar(email, None)
                    st.success("Avatar removed."); time.sleep(0.5); st.rerun()

    with tab2:
        st.markdown("#### Update Email Address")
        st.info(f"Current email: **{email}**")
        new_email        = st.text_input("New Email Address",             key="new_email_input")
        confirm_email    = st.text_input("Confirm New Email",             key="confirm_email_input")
        current_pw_email = st.text_input("Current Password (to confirm)", type="password", key="pw_for_email")
        if st.button("📧  Update Email", type="primary", key="update_email_btn"):
            if not new_email or not confirm_email or not current_pw_email:
                st.error("All fields are required.")
            elif not is_valid_email(new_email): st.error("Please enter a valid email address.")
            elif new_email != confirm_email: st.error("Email addresses do not match.")
            elif new_email.lower() == email.lower(): st.warning("New email is the same as current email.")
            else:
                ok, _ = authenticate_user(email, current_pw_email)
                if not ok: st.error("Incorrect password.")
                elif check_user_exists_by_email(new_email.lower()): st.error("This email is already registered.")
                else:
                    success, msg = update_user_email(email, new_email.lower())
                    if success:
                        token = create_access_token({"sub": new_email.lower(), "username": username})
                        st.session_state["jwt_token"] = token
                        st.success("✅ Email updated! Please log in again.")
                        time.sleep(1.5); st.session_state.update({"jwt_token":None,"page":"login"}); st.rerun()
                    else: st.error(f"Update failed: {msg}")

    with tab3:
        st.markdown("#### Change Password")
        cur_pw  = st.text_input("Current Password",    type="password", key="prof_cur_pw")
        new_pw  = st.text_input("New Password",        type="password", key="prof_new_pw")
        if new_pw:
            strength, fb = check_password_strength(new_pw)
            color = {"Strong":"#8E4585","Medium":"#FACC15","Weak":"#ef4444"}[strength]
            st.markdown(f'<div style="font-size:13px;margin-bottom:6px;">Strength: <span style="color:{color};font-weight:600;">{strength}</span></div>', unsafe_allow_html=True)
        conf_pw = st.text_input("Confirm New Password", type="password", key="prof_conf_pw")
        if st.button("🔑  Update Password", type="primary", key="prof_update_pw"):
            if not cur_pw or not new_pw or not conf_pw: st.error("All fields are required.")
            elif new_pw != conf_pw: st.error("New passwords do not match.")
            else:
                strength, fb = check_password_strength(new_pw)
                if strength == "Weak": st.error(f"Weak password: {', '.join(fb)}")
                else:
                    ok, _ = authenticate_user(email, cur_pw)
                    if not ok: st.error("Current password is incorrect.")
                    elif check_password_reused(email, new_pw): st.warning("⚠️ This password was used before.")
                    else:
                        update_password(email, new_pw)
                        st.success("✅ Password changed! Please log in again.")
                        time.sleep(1.5); st.session_state.update({"jwt_token":None,"page":"login"}); st.rerun()

    with tab4:
        st.markdown("#### Share Your Feedback")
        feat_opts  = ["General","Readability","Summarize","Paraphrase","Augment","History","Admin Dashboard","UI/UX"]
        fb_feature = st.selectbox("Feature", feat_opts, key="fb_feature")
        fb_rating  = st.slider("Rating ⭐", 1, 5, 4, key="fb_rating")
        stars = "⭐" * fb_rating + "☆" * (5 - fb_rating)
        st.markdown(f"<div style='font-size:22px;margin:4px 0 8px;'>{stars}</div>", unsafe_allow_html=True)
        fb_comment = st.text_area("Comments (optional)", height=120, key="fb_comment")
        if st.button("📤  Submit Feedback", type="primary", key="submit_fb"):
            save_feedback(username, fb_rating, fb_feature, fb_comment)
            st.success("✅ Thank you for your feedback!"); time.sleep(0.5); st.rerun()


# ══════════════════════════════════════════════════════════
# ADMIN PAGE
# ══════════════════════════════════════════════════════════
def admin_page():
    payload = verify_token(st.session_state["jwt_token"])
    if not payload: st.session_state.update({"jwt_token":None,"page":"login"}); st.rerun(); return
    _admin_uname = payload.get("username","").lower()
    _admin_email = payload.get("sub","")
    if not (_admin_uname == "admin" or get_is_admin_from_db(_admin_email)):
        st.error("⛔ Access Denied"); return

    page_header("🛡️", "Admin Command Center", "Monitor, manage, and analyze platform activity")

    users    = get_all_users()
    all_hist = get_all_history_from_db(500)
    all_fb   = get_all_feedback()

    total_users  = len(users)
    def _safe_today(u):
        dt = u[2]
        if not dt or isinstance(dt, bool): return False
        try:
            d = dt.replace(tzinfo=None) if hasattr(dt,'tzinfo') and dt.tzinfo else dt
            return (datetime.datetime.utcnow() - d).days < 1
        except: return False
    today_joins  = sum(1 for u in users if _safe_today(u))
    locked_count = sum(1 for u in users if u[4])
    admin_count  = sum(1 for u in users if u[3])

    m1,m2,m3,m4,m5 = st.columns(5)
    m1.metric("👥 Total Users",   total_users)
    m2.metric("🛡️ Admins",       admin_count)
    m3.metric("🔒 Locked",        locked_count)
    m4.metric("📅 Today Joins",   today_joins)
    m5.metric("📋 Total Actions", len(all_hist))

    st.markdown("---")

    tab_users, tab_hist, tab_analytics, tab_feedback, tab_export, tab_conv = st.tabs([
        "👥  Users","📋  System History","📊  Analytics","💬  Feedback","⬇️  Export","🗨️  Conversations"
    ])

    with tab_users:
        st.markdown("#### User Management")
        search_q = st.text_input("🔍 Search by username or email", key="admin_search")
        COL_W = [1.5, 2.8, 1.0, 1.0, 0.75, 0.6]
        filtered_users = [u for u in users if not search_q or search_q.lower() in u[0].lower() or search_q.lower() in u[1].lower()]

        for uname, uemail, ucreated, is_adm, is_lock in filtered_users:
            st.markdown('<div style="height:1px;background:#C4B5FD;margin:0;opacity:0.4;"></div>', unsafe_allow_html=True)
            c1,c2,c3,c4,c5,c6 = st.columns(COL_W)
            badge     = " 🛡️" if is_adm else ""
            lock_icon = "🔒" if is_lock else "🔓"
            c1.markdown(f'<div style="padding:6px 0;font-weight:600;font-size:13px;">{uname}{badge}</div>', unsafe_allow_html=True)
            c2.markdown(f'<div style="padding:6px 0;color:#6B7280;font-size:12px;">{uemail}</div>', unsafe_allow_html=True)
            c3.markdown(f'<div style="padding:6px 0;color:#A78BFA;font-size:12px;">{get_relative_time(ucreated) if ucreated else "N/A"}</div>', unsafe_allow_html=True)
            if uname.lower() == "admin":
                c4.markdown('<div style="padding:6px 0;text-align:center;color:#6D28D9;">🛡️ Root</div>', unsafe_allow_html=True)
                c5.markdown('<div style="padding:6px 0;text-align:center;color:#6B7280;">—</div>', unsafe_allow_html=True)
                c6.markdown('<div style="padding:6px 0;text-align:center;color:#6B7280;">—</div>', unsafe_allow_html=True)
            else:
                admin_label = "➖ Demote" if is_adm else "⬆️ Promote"
                if c4.button(admin_label, key=f"adm_{uemail}", use_container_width=True):
                    set_admin_status(uemail, not is_adm); st.rerun()
                if c5.button(lock_icon, key=f"lck_{uemail}", use_container_width=True):
                    set_locked_status(uemail, not is_lock); st.rerun()
                if c6.button("🗑️", key=f"del_{uemail}", use_container_width=True):
                    try:
                        delete_user(uemail)
                        st.warning(f"✅ Deleted: {uemail}"); time.sleep(0.3); st.rerun()
                    except Exception as e:
                        st.error(f"❌ Delete failed: {e}")
        if not filtered_users: st.info("No users found.")

    with tab_hist:
        st.markdown("#### Full System Activity Log")
        if all_hist:
            rows = [{"User":r[0],"Action":r[1],"Model":r[2],"Language":r[6],"Words In":r[7],"Words Out":r[8],"Date":str(r[11])[:19]} for r in all_hist]
            st.dataframe(pd.DataFrame(rows), use_container_width=True, height=450)
        else:
            st.info("No activity recorded yet.")

    with tab_analytics:
        if not all_hist:
            st.info("Not enough data for analytics yet.")
        else:
            from collections import Counter
            actions = [r[1] for r in all_hist if r[1]]
            models  = [r[2] for r in all_hist if r[2]]
            langs   = [r[6] for r in all_hist if r[6]]
            col_a, col_b = st.columns(2)
            with col_a:
                st.markdown("##### 🔧 Feature Usage")
                act_counts = Counter(actions)
                fig_feat = go.Figure(go.Bar(x=list(act_counts.keys()), y=list(act_counts.values()), marker_color=["#8B5CF6","#8E4585","#A78BFA"], text=list(act_counts.values()), textposition="outside"))
                fig_feat.update_layout(paper_bgcolor="#FFFFFF", plot_bgcolor="#F1F5F9", font=dict(color="#6B7280",family="Inter"), height=300, margin=dict(l=10,r=10,t=30,b=10))
                st.plotly_chart(fig_feat, use_container_width=True)
            with col_b:
                st.markdown("##### 🌐 Language Distribution")
                lang_counts = Counter(langs)
                fig_lang = go.Figure(go.Pie(labels=list(lang_counts.keys()), values=list(lang_counts.values()), hole=0.45, marker=dict(colors=["#8B5CF6","#A78BFA","#6D28D9","#C4B5FD","#8E4585","#DDD6FE","#6B7280"])))
                fig_lang.update_layout(paper_bgcolor="#FFFFFF", font=dict(color="#6B7280",family="Inter"), height=300, margin=dict(l=10,r=10,t=30,b=10))
                st.plotly_chart(fig_lang, use_container_width=True)

    with tab_feedback:
        st.markdown("#### User Feedback Analysis")
        if not all_fb:
            st.info("No feedback submitted yet.")
        else:
            ratings  = [f[1] for f in all_fb]
            features = [f[2] for f in all_fb]
            comments = [f[3] for f in all_fb if f[3] and len(f[3].strip()) > 2]
            avg_rating = round(sum(ratings)/len(ratings), 2) if ratings else 0
            m1,m2,m3 = st.columns(3)
            m1.metric("📝 Total Responses", len(all_fb))
            m2.metric("⭐ Avg Rating",       f"{avg_rating}/5")
            m3.metric("💬 With Comments",    len(comments))
            from collections import Counter
            col_fa, col_fb2 = st.columns(2)
            with col_fa:
                rc = Counter(ratings)
                fig_r = go.Figure(go.Bar(x=[f"{i}★" for i in range(1,6)], y=[rc.get(i,0) for i in range(1,6)], marker_color=["#6D28D9","#8B5CF6","#A78BFA","#C4B5FD","#8E4585"]))
                fig_r.update_layout(paper_bgcolor="#FFFFFF", plot_bgcolor="#F1F5F9", font=dict(color="#6B7280",family="Inter"), height=260, margin=dict(l=10,r=10,t=20,b=10))
                st.plotly_chart(fig_r, use_container_width=True)
            with col_fb2:
                fc = Counter(features)
                fig_f = go.Figure(go.Pie(labels=list(fc.keys()), values=list(fc.values()), hole=0.4, marker=dict(colors=["#8B5CF6","#A78BFA","#6D28D9","#C4B5FD","#8E4585","#DDD6FE","#6B7280","#EDE9FE"])))
                fig_f.update_layout(paper_bgcolor="#FFFFFF", font=dict(color="#6B7280",family="Inter"), height=260, margin=dict(l=10,r=10,t=20,b=10))
                st.plotly_chart(fig_f, use_container_width=True)
            if comments and WORDCLOUD_AVAILABLE:
                try:
                    text = " ".join(comments)
                    wc = WordCloud(width=900, height=350, background_color="#F1F5F9", colormap="Purples", max_words=100).generate(text)
                    fig_wc, ax = plt.subplots(figsize=(11, 4))
                    fig_wc.patch.set_facecolor("#F1F5F9")
                    ax.set_facecolor("#F1F5F9")
                    ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
                    st.pyplot(fig_wc, use_container_width=True)
                    plt.close(fig_wc)
                except Exception as e:
                    st.warning(f"WordCloud error: {e}")
            with st.expander("📋 Raw Feedback Data"):
                fb_rows = [{"User":f[0],"Rating":str(f[1])+"★","Feature":f[2],"Comment":f[3] or ""} for f in all_fb]
                st.dataframe(pd.DataFrame(fb_rows), use_container_width=True, hide_index=True)

    with tab_export:
        st.markdown("#### Download Platform Data")
        col_e1, col_e2, col_e3 = st.columns(3)
        for col, icon, label, desc, fname, data in [
            (col_e1,"👥","User Details","All registered accounts","users.csv",
             pd.DataFrame([{"Username":u[0],"Email":u[1],"Joined":str(u[2])[:19],"Is Admin":u[3],"Is Locked":u[4]} for u in users]).to_csv(index=False)),
            (col_e2,"📋","Activity History","All system usage logs","history.csv",
             pd.DataFrame([{"User":r[0],"Action":r[1],"Model":r[2],"Language":r[6],"Words In":r[7],"Words Out":r[8],"Date":str(r[11])[:19]} for r in all_hist]).to_csv(index=False) if all_hist else "No data"),
            (col_e3,"💬","User Feedback","All ratings and comments","feedback.csv",
             pd.DataFrame([{"User":f[0],"Rating":str(f[1])+" stars","Feature":f[2],"Comment":f[3] or ""} for f in all_fb]).to_csv(index=False) if all_fb else "No data"),
        ]:
            with col:
                st.markdown(f"""<div style="background:rgba(139,92,246,0.07);border:1px solid #C4B5FD;border-radius:12px;padding:20px;text-align:center;margin-bottom:12px;">
                    <div style="font-size:32px;margin-bottom:8px;">{icon}</div>
                    <div style="font-weight:600;color:#6D28D9;margin-bottom:4px;">{label}</div>
                    <div style="font-size:12px;color:#6B7280;">{desc}</div>
                </div>""", unsafe_allow_html=True)
                st.download_button(f"⬇️ Download {label} CSV", data, fname, "text/csv", use_container_width=True)

    with tab_conv:
        st.markdown("#### 🗨️ User Chat Conversations")
        chat_users = get_all_chat_users()
        if not chat_users:
            st.info("No chat conversations yet.")
        else:
            selected_user = st.selectbox("Select a user", chat_users, key="admin_conv_user")
            if selected_user:
                conv = get_chat_history_admin(selected_user)
                if not conv: st.info(f"No messages from {selected_user}.")
                else:
                    cm1,cm2,cm3 = st.columns(3)
                    cm1.metric("💬 Total Messages", len(conv))
                    cm2.metric("👤 User Messages",  sum(1 for r in conv if r[0]=="user"))
                    cm3.metric("🤖 AI Responses",   sum(1 for r in conv if r[0]=="assistant"))
                    for role, message, model, created_at in conv:
                        ts = str(created_at)[:19] if created_at else ""
                        if role == "user":
                            st.markdown(f"""<div style="display:flex;justify-content:flex-end;margin:6px 0;">
                                <div><div style="text-align:right;font-size:10px;color:#6B7280;margin-bottom:3px;">{ts}</div>
                                <div style="background:linear-gradient(135deg,#8B5CF6,#A78BFA);color:#fff;border-radius:16px 16px 4px 16px;padding:10px 16px;max-width:600px;font-size:13px;display:inline-block;">{message}</div></div>
                                <div style="font-size:20px;margin-left:8px;align-self:flex-end;">👤</div>
                            </div>""", unsafe_allow_html=True)
                        else:
                            st.markdown(f"""<div style="display:flex;justify-content:flex-start;margin:6px 0;">
                                <div style="font-size:20px;margin-right:8px;align-self:flex-end;">🤖</div>
                                <div><div style="font-size:10px;color:#6B7280;margin-bottom:3px;">{model} · {ts}</div>
                                <div style="background:#FFFFFF;border:1px solid #C4B5FD;color:#1F2937;border-radius:16px 16px 16px 4px;padding:10px 16px;max-width:600px;font-size:13px;display:inline-block;">{message}</div></div>
                            </div>""", unsafe_allow_html=True)
                    conv_rows = [{"Role":r[0],"Message":r[1],"Model":r[2],"Time":str(r[3])[:19]} for r in conv]
                    st.download_button(f"⬇️ Download {selected_user}'s Conversation", pd.DataFrame(conv_rows).to_csv(index=False), f"chat_{selected_user}.csv", "text/csv")


# ══════════════════════════════════════════════════════════
# DASHBOARD SHELL
# ══════════════════════════════════════════════════════════
def dashboard_page():
    token   = st.session_state["jwt_token"]
    payload = verify_token(token)
    if not payload: st.session_state.update({"jwt_token":None,"page":"login"}); st.rerun(); return
    username = payload.get("username","User")
    email    = payload.get("sub","")
    is_admin = (username.lower() == "admin") or get_is_admin_from_db(email)
    default_nav = "Admin" if is_admin else "Readability"
    if "nav_selected" not in st.session_state:
        st.session_state["nav_selected"] = default_nav

    admin_accessible = {"Admin", "History"}
    user_only_pages  = {"Readability","Summarize","Paraphrase","Augment","Chat","Profile"}

    if is_admin and st.session_state["nav_selected"] in user_only_pages:
        st.session_state["nav_selected"] = "Admin"; st.rerun()
    if not is_admin and st.session_state["nav_selected"] not in (user_only_pages | {"History"}):
        st.session_state["nav_selected"] = "Readability"; st.rerun()

    with st.sidebar:
        initials = username[0].upper() if username else "U"
        if is_admin:
            avatar_html = (f'<div style="width:56px;height:56px;border-radius:50%;background:linear-gradient(135deg,#6D28D9,#8B5CF6);display:flex;align-items:center;justify-content:center;font-size:26px;color:#fff;margin:0 auto 12px;">🛡️</div>')
        else:
            avatar_b64 = get_user_avatar(email)
            if avatar_b64:
                avatar_html = (f'<img src="data:image/png;base64,{avatar_b64}" style="width:56px;height:56px;border-radius:50%;object-fit:cover;border:2px solid #8B5CF6;margin:0 auto 12px;display:block;" />')
            else:
                avatar_html = (f'<div style="width:56px;height:56px;border-radius:50%;background:linear-gradient(135deg,#8B5CF6,#A78BFA);display:flex;align-items:center;justify-content:center;font-size:24px;font-weight:700;color:#fff;margin:0 auto 12px;">{initials}</div>')
        st.markdown(f"""<div class="sidebar-logo">
            {avatar_html}
            <div class="app-name">TEXTMORPH</div>
            <div class="user-name">✔️ {username}</div>
            <div class="user-email">{email}</div>
        </div>""", unsafe_allow_html=True)

        st.markdown('<div style="height:2px;background:linear-gradient(90deg,#7C3AED,#C4B5FD,#7C3AED);border-radius:2px;margin:4px 8px 12px;"></div>', unsafe_allow_html=True)

        if is_admin:
            nav_items = [("🛡️  Admin Dashboard","Admin"),("🕘  History","History")]
        else:
            nav_items = [
                ("📊  Readability","Readability"),("📝  Summarize","Summarize"),
                ("🔄  Paraphrase","Paraphrase"),("🗃️  Augment","Augment"),
                ("💬  Chat","Chat"),("🕘  History","History"),("👤  Profile","Profile"),
            ]

        menu_keys  = [key for _, key in nav_items]
        menu_icons = {"Readability":"bar-chart-fill","Summarize":"file-text","Paraphrase":"arrow-repeat",
                      "Augment":"database","Chat":"chat-dots-fill","History":"clock-history",
                      "Profile":"person-fill","Admin":"shield-fill"}
        cur_nav = st.session_state.get("nav_selected", default_nav)
        cur_idx = menu_keys.index(cur_nav) if cur_nav in menu_keys else 0

        _is_dark = st.session_state.get("theme","dark") == "dark"
        _nav_bg       = "#212121" if _is_dark else "#FFFFFF"
        _nav_text     = "#C4B5FD" if _is_dark else "#374151"
        _nav_sel_bg   = "linear-gradient(135deg,#2D1F55,#3B2D6B)" if _is_dark else "linear-gradient(135deg,#EDE9FE,#DDD6FE)"
        _nav_sel_text = "#E2D9F3" if _is_dark else "#6D28D9"

        nav_sel = option_menu(
            menu_title=None,
            options=menu_keys,
            icons=[menu_icons.get(k,"circle") for k in menu_keys],
            default_index=cur_idx,
            styles={
                "container":         {"padding":"0","background-color": _nav_bg},
                "icon":              {"color":"#A78BFA","font-size":"15px"},
                "nav-link":          {"font-size":"14px","text-align":"left","margin":"2px 0",
                                      "color": _nav_text,"border-radius":"10px","padding":"10px 16px"},
                "nav-link-selected": {"background": _nav_sel_bg,"color": _nav_sel_text,
                                      "font-weight":"600","border-left":"3px solid #8B5CF6"},
            },
        )
        if nav_sel != st.session_state.get("nav_selected", default_nav):
            st.session_state["nav_selected"] = nav_sel; st.rerun()

        st.markdown('<div style="height:2px;background:linear-gradient(90deg,#7C3AED,#C4B5FD,#7C3AED);border-radius:2px;margin:12px 8px 8px;"></div>', unsafe_allow_html=True)

        col_th, col_so = st.columns([1,1])
        with col_th:
            is_light = st.session_state.get("theme","dark") == "light"
            theme_icon = "🌙" if is_light else "☀️"
            if st.button(theme_icon, use_container_width=True, type="secondary", help="Toggle theme"):
                st.session_state["theme"] = "light" if st.session_state.get("theme","dark")=="dark" else "dark"
                st.rerun()
        with col_so:
            if st.button("🔓", use_container_width=True, type="secondary", help="Sign Out"):
                st.session_state.update({"jwt_token":None,"username":None,"nav_selected":"Readability","page":"login"})
                st.rerun()
        theme_txt = "☀️ Light Mode" if st.session_state.get("theme","dark")=="light" else "🌙 Dark Mode"
        st.markdown(f'<div style="text-align:center;font-size:11px;color:#9C8BBF;margin-top:4px;">{theme_txt}</div>', unsafe_allow_html=True)

    sel = st.session_state["nav_selected"]
    if   sel == "Admin":       admin_page()
    elif sel == "Readability": readability_page()
    elif sel == "Summarize":   summarizer_page()
    elif sel == "Paraphrase":  paraphraser_page()
    elif sel == "Augment":     augmentation_page()
    elif sel == "Chat":        chat_page()
    elif sel == "History":     history_page(username, is_admin=is_admin)
    elif sel == "Profile":     profile_page()


# ══════════════════════════════════════════════════════════
# NUCLEAR BUTTON / TAB FIX CSS
# ══════════════════════════════════════════════════════════
st.markdown("""
<style>
button, .stButton > button,
button[data-testid="stBaseButton-primary"],
div[data-testid="stButton"] button,
button[kind="primary"] {
  background: linear-gradient(135deg, #8E4585, #6D28D9) !important;
  background-color: #6D28D9 !important;
  color: #FFFFFF !important;
  border: none !important;
}
div[data-testid="stTextInput"] button,
div[data-baseweb="input"] button,
div[data-baseweb="base-input"] button {
  background: transparent !important;
  background-color: transparent !important;
  border: none !important;
  box-shadow: none !important;
  color: #6B7280 !important;
}
.stTabs [data-baseweb="tab"],
.stTabs [data-baseweb="tab"][aria-selected="false"],
div[data-baseweb="tab-list"] button {
  background: transparent !important;
  color: #6B7280 !important;
  border-bottom: none !important;
}
.stTabs [aria-selected="true"],
div[data-baseweb="tab-list"] button[aria-selected="true"] {
  background: #EDE9FE !important;
  color: #6D28D9 !important;
  border-bottom: 2px solid #8B5CF6 !important;
}
[data-testid="stSlider"] [role="slider"] {
  background-color: #7C3AED !important;
  border-color: #6D28D9 !important;
}
</style>
""", unsafe_allow_html=True)

# ══════════════════════════════════════════════════════════
# ROUTER
# ══════════════════════════════════════════════════════════
if st.session_state.get("jwt_token"):
    payload = verify_token(st.session_state["jwt_token"])
    if payload: dashboard_page()
    else: st.session_state.update({"jwt_token":None,"page":"login"}); st.rerun()
else:
    page = st.session_state.get("page","login")
    if   page == "signup": signup_page()
    elif page == "forgot": forgot_password_page()
    else:                  login_page()


Overwriting app.py


In [ ]:
!pip install pyngrok==6.1.2 -q

In [ ]:
import os
from google.colab import userdata

NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')

os.environ['EMAIL_PASSWORD']  = userdata.get('EMAIL_PASSWORD')
os.environ['EMAIL_ID']        = userdata.get('EMAIL_ID')
EMAIL_ADDRESS = os.environ.get("EMAIL_ID").strip()
os.environ['ADMIN_EMAIL_ID']  = userdata.get('ADMIN_EMAIL_ID')
os.environ['ADMIN_PASSWORD']  = userdata.get('ADMIN_PASSWORD')
os.environ['JWT_SECRET']      = 'super-secret-change-me'
os.environ['DB_NAME']         = 'milestone1_users'
os.environ['DB_USER']         = 'postgres'

print("✅ Environment variables set")

✅ Environment variables set


## 8. Launch

In [ ]:
import os, subprocess, time, json

from google.colab import userdata

# ── Environment Variables ──────────────────────────────────────
NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')

os.environ['EMAIL_PASSWORD']  = userdata.get('EMAIL_PASSWORD')
os.environ['EMAIL_ID']        = userdata.get('EMAIL_ID')
os.environ['ADMIN_EMAIL_ID']  = userdata.get('ADMIN_EMAIL_ID')
os.environ['ADMIN_PASSWORD']  = userdata.get('ADMIN_PASSWORD')
os.environ['JWT_SECRET']      = 'super-secret-change-me'
os.environ['DB_NAME']         = 'milestone1_users'
os.environ['DB_USER']         = 'postgres'

# ── Install ngrok ──────────────────────────────────────────────
subprocess.run("curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null", shell=True)
subprocess.run('echo "deb https://ngrok-agent.s3.amazonaws.com buster main" | sudo tee /etc/apt/sources.list.d/ngrok.list', shell=True)
subprocess.run("sudo apt update -q && sudo apt install ngrok -q", shell=True)

# ── Authenticate ngrok ─────────────────────────────────────────
subprocess.run(f"ngrok authtoken {NGROK_AUTHTOKEN}", shell=True)

# ── Start Streamlit FIRST ──────────────────────────────────────
subprocess.Popen("streamlit run app.py --server.port 8501 --server.headless true", shell=True)
time.sleep(5)

# ── Start ngrok AFTER Streamlit ────────────────────────────────
subprocess.Popen("ngrok http 8501", shell=True)

# ── Retry until tunnel URL is ready ───────────────────────────
for i in range(10):
    result = subprocess.run("curl -s http://localhost:4040/api/tunnels", shell=True, capture_output=True, text=True)
    if result.stdout.strip():
        data = json.loads(result.stdout)
        if data.get("tunnels"):
            url = data['tunnels'][0]['public_url']
            print(f"🌐 App is live at: {url}")
            break
    print(f"⏳ Attempt {i+1}: waiting for ngrok...")
    time.sleep(2)
else:
    print("❌ ngrok didn't start. Check token or increase retries.")

🌐 App is live at: https://nonfavorably-untrying-li.ngrok-free.dev


In [ ]:
# ─────────────────────────────────────────────
# Install cloudflared
# ─────────────────────────────────────────────
""""import subprocess

subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared", shell=True)
subprocess.run("chmod +x cloudflared", shell=True)

print("✅ cloudflared ready")

In [ ]:
# ─────────────────────────────────────────────
# Launch Cloudflare tunnel
# ─────────────────────────────────────────────
""""import subprocess, time, re, urllib.request

PORT = 8501

# Start Streamlit
subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", str(PORT),
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)  # give Streamlit more time to boot
print("✅ Streamlit running on port", PORT)

# Start Cloudflare tunnel
cf = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print("🔄 Waiting for Cloudflare tunnel URL...")
url = None
deadline = time.time() + 40

while time.time() < deadline:
    line = cf.stdout.readline().decode("utf-8", errors="ignore")
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        print(f"✅ Tunnel URL found: {url}")
        break

if not url:
    print("❌ Could not get tunnel URL. Re-run this cell.")
else:
    # ── Wait until URL is actually reachable ──
    print("⏳ Waiting for DNS to propagate...")
    for attempt in range(20):  # try every 3 seconds, up to 60s
        try:
            urllib.request.urlopen(url, timeout=5)
            print(f"\n{'='*50}")
            print(f"🌐  App is live at:  {url}")
            print(f"{'='*50}\n")
            break
        except Exception:
            print(f"   attempt {attempt+1}/20 — not ready yet...")
            time.sleep(3)
    else:
        print(f"\n⚠️  DNS still not ready, but try opening manually:")
        print(f"   {url}")
        print("   (wait 10 more seconds and refresh)")

In [ ]:
# ── STOP SERVER ──────────────────────────────────────────
"""import subprocess, builtins
from pyngrok import ngrok

ngrok.kill()
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)

if hasattr(builtins, '_st_proc'):
    builtins._st_proc.terminate()

print('✅ Server stopped.')

## 9. Backup DB

In [ ]:
# ── Save PostgreSQL database to Google Drive ─────────────────
# Run this cell anytime to backup your DB to Drive
import subprocess, os
BACKUP_PATH = '/content/drive/MyDrive/postgres_milestone_db/milestone1_users.sql'
os.makedirs(os.path.dirname(BACKUP_PATH), exist_ok=True)
result = subprocess.run(
    f'sudo -u postgres pg_dump milestone1_users > "{BACKUP_PATH}"',
    shell=True, capture_output=True, text=True
)
if result.returncode == 0:
    size = os.path.getsize(BACKUP_PATH)
    print(f'✅ Database saved to Google Drive! ({size} bytes)')
    print(f'📁 Path: {BACKUP_PATH}')
else:
    print(f'❌ Backup failed: {result.stderr}')


✅ Database saved to Google Drive! (22807 bytes)
📁 Path: /content/drive/MyDrive/postgres_milestone_db/milestone1_users.sql
